In [ ]:
#!git clone https://github.com/chen700564/RGB.git

In [ ]:
# ==============================================================================
# SECTION 0: Environment Setup, Drive Mount, Global Configuration
# ==============================================================================

import os
import sys
import json
import time
import logging
from typing import Dict, List, Any

# ------------------------------------------------------------------------------
# 0.1 Install required packages
# ------------------------------------------------------------------------------

!pip install -q groq pyyaml tqdm

# ------------------------------------------------------------------------------
# 0.2 Google Drive Mount
# ------------------------------------------------------------------------------

from google.colab import drive

DRIVE_MOUNT_POINT = "/content/drive"
drive.mount(DRIVE_MOUNT_POINT)

# ------------------------------------------------------------------------------
# 0.3 Project Root Configuration
# ------------------------------------------------------------------------------

#PROJECT_ROOT = os.path.join(
 #   DRIVE_MOUNT_POINT,
 #   "MyDrive",
#  "cap0401_rgb_rag_evaluation"
#)
PROJECT_ROOT = os.path.join(
    DRIVE_MOUNT_POINT,
    "MyDrive"
)

DATASET_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results_01")
LOG_DIR = os.path.join(PROJECT_ROOT, "logs")
CONFIG_DIR = os.path.join(PROJECT_ROOT, "config")

for path in [PROJECT_ROOT, DATASET_DIR, RESULTS_DIR, LOG_DIR, CONFIG_DIR]:
    os.makedirs(path, exist_ok=True)

# ------------------------------------------------------------------------------
# 0.4 Logging Configuration
# ------------------------------------------------------------------------------

LOG_FILE_PATH = os.path.join(LOG_DIR, "execution_debug.log")

logger = logging.getLogger("cap_25_rgbeval")
logger.setLevel(logging.DEBUG)

logger.handlers.clear()

file_handler = logging.FileHandler(LOG_FILE_PATH)
file_handler.setLevel(logging.DEBUG)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

formatter = logging.Formatter(
    fmt="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("Logger initialized")
logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Dataset directory: %s", DATASET_DIR)
logger.info("Results directory: %s", RESULTS_DIR)

# ------------------------------------------------------------------------------
# 0.5 Global Experiment Configuration
# ------------------------------------------------------------------------------

EXPERIMENT_CONFIG: Dict[str, Any] = {
    "project_name": "rgb_evaluation",
    "language": "en",
    "passage_num": 5,
    "noise_rates": [0.0, 0.2, 0.4, 0.6, 0.8],
    "temperature": 0.0,
    "max_retries": 3,
    "retry_backoff_sec": 5,
    "tasks": {
        "noise_robustness": {
            "dataset": "en_refine.json"
        },
        "negative_rejection": {
            "dataset": "en_refine.json"
        },
        "information_integration": {
            "dataset": "en_int.json"
        },
        "counterfactual_robustness": {
            "dataset": "en_fact.json"
        }
    }
}

logger.info("Experiment configuration loaded")

# ------------------------------------------------------------------------------
# 0.6 Groq Model Registry
# ------------------------------------------------------------------------------
GROQ_MODELS: Dict[str, str] = {
    "qwen3_32b": "qwen/qwen3-32b",
    #"llama3_1_8b": "llama-3.1-8b-instant",
    #"llama3_3_70b": "llama-3.3-70b-versatile",
    #"gpt_oss_20b": "openai/gpt-oss-20b",
    #"gpt_oss_120b": "openai/gpt-oss-120b",
    #"kimi_k2" : "moonshotai/kimi-k2-instruct-0905",
    #"whisper" : "canopylabs/orpheus-v1-english",
    #"gpt-oss-safeguard" : "openai/gpt-oss-safeguard-20b"
}

logger.info("Registered Groq models: %s", list(GROQ_MODELS.keys()))

# ------------------------------------------------------------------------------
# 0.7 Groq Client Initialization
# ------------------------------------------------------------------------------

from groq import Groq

def initialize_groq_client(api_key: str) -> Groq:
    if not api_key or not isinstance(api_key, str):
        raise ValueError("Invalid Groq API key provided")

    logger.info("Initializing Groq client")
    client = Groq(api_key=api_key)
    logger.info("Groq client initialized successfully")
    return client

# ------------------------------------------------------------------------------
# 0.8 Utility: Safe JSON Load
# ------------------------------------------------------------------------------

def load_json_from_drive(file_path: str) -> Any:
    if not os.path.exists(file_path):
        logger.error("JSON file not found: %s", file_path)
        raise FileNotFoundError(file_path)

    logger.debug("Loading JSON file: %s", file_path)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    logger.debug("Loaded JSON entries: %d", len(data) if isinstance(data, list) else 1)
    return data

    logger.info(
    "Noise=%.1f | Clean=%d | Noise=%d",
    noise,
    sum(d == inst["documents"][0] for _, d in docs),
    len(docs) - sum(d == inst["documents"][0] for _, d in docs)
    )


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.5 MB/s eta 0:00:00
Mounted at /content/drive
2026-01-24 21:48:32 | INFO | Logger initialized


INFO:cap_25_rgbeval:Logger initialized


2026-01-24 21:48:32 | INFO | Project root: /content/drive/MyDrive


INFO:cap_25_rgbeval:Project root: /content/drive/MyDrive


2026-01-24 21:48:32 | INFO | Dataset directory: /content/drive/MyDrive/data


...

INFO:cap_25_rgbeval:Results directory: /content/drive/MyDrive/results_01


2026-01-24 21:48:32 | INFO | Experiment configuration loaded


INFO:cap_25_rgbeval:Experiment configuration loaded


2026-01-24 21:48:32 | INFO | Registered Groq models: ['qwen3_32b']


INFO:cap_25_rgbeval:Registered Groq models: ['qwen3_32b']


In [ ]:
# ==============================================================================
# SECTION 1: Dataset Management and Validation
# ==============================================================================

import os
import json
from typing import List, Dict, Any

# ------------------------------------------------------------------------------
# 1.1 Dataset Path Registry
# ------------------------------------------------------------------------------

DATASET_PATHS: Dict[str, str] = {
    "en_refine": os.path.join(DATASET_DIR, "en_refine.json"),
    "en_int": os.path.join(DATASET_DIR, "en_int.json"),
    "en_fact": os.path.join(DATASET_DIR, "en_fact.json"),
}

logger.info("Dataset paths registered")

# ------------------------------------------------------------------------------
# 1.2 Dataset Existence Check
# ------------------------------------------------------------------------------

def validate_dataset_files(dataset_paths: Dict[str, str]) -> None:
    missing_files = []

    for name, path in dataset_paths.items():
        if not os.path.exists(path):
            logger.error("Missing dataset file: %s", path)
            missing_files.append(path)
        else:
            logger.info("Found dataset file: %s", path)

    if missing_files:
        raise FileNotFoundError(
            "Dataset staging incomplete. Place all required RGB datasets in Drive."
        )

validate_dataset_files(DATASET_PATHS)

# ------------------------------------------------------------------------------
# 1.3 JSONL Loader
# ------------------------------------------------------------------------------

def load_jsonl(file_path: str) -> List[Dict[str, Any]]:
    records = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                logger.error(
                    "JSON parse error in file %s at line %d",
                    file_path,
                    line_no
                )
                raise e

    return records

# ------------------------------------------------------------------------------
# 1.4 Dataset Schema Validation Utilities
# ------------------------------------------------------------------------------

def validate_common_fields(instance: Dict[str, Any]) -> None:
    required_fields = ["id", "query", "answer", "positive", "negative"]

    for field in required_fields:
        if field not in instance:
            raise ValueError(f"Missing required field: {field}")

    if not isinstance(instance["positive"], list):
        raise TypeError("Field 'positive' must be a list")

    if not isinstance(instance["negative"], list):
        raise TypeError("Field 'negative' must be a list")


def validate_information_integration_fields(instance: Dict[str, Any]) -> None:
    for group in instance["positive"]:
        if not isinstance(group, list):
            raise TypeError(
                "Information integration requires 'positive' as list of lists"
            )


def validate_counterfactual_fields(instance: Dict[str, Any]) -> None:
    if "positive_wrong" not in instance:
        raise ValueError("Missing 'positive_wrong' in counterfactual dataset")

# ------------------------------------------------------------------------------
# 1.5 Dataset Loader
# ------------------------------------------------------------------------------

def load_and_validate_dataset(
    dataset_key: str,
    dataset_path: str
) -> List[Dict[str, Any]]:

    logger.info("Loading dataset: %s", dataset_key)

    data = load_jsonl(dataset_path)

    logger.info(
        "Dataset %s loaded with %d records",
        dataset_key,
        len(data)
    )

    for idx, instance in enumerate(data):
        try:
            validate_common_fields(instance)

            if dataset_key == "en_int":
                validate_information_integration_fields(instance)

            if dataset_key == "en_fact":
                validate_counterfactual_fields(instance)

        except Exception as e:
            logger.error(
                "Schema validation failed for dataset=%s index=%d id=%s",
                dataset_key,
                idx,
                instance.get("id", "UNKNOWN")
            )
            raise e

    logger.info("Schema validation successful for dataset: %s", dataset_key)
    return data

# ------------------------------------------------------------------------------
# 1.6 Load All Datasets
# ------------------------------------------------------------------------------

DATASETS: Dict[str, List[Dict[str, Any]]] = {}

for key, path in DATASET_PATHS.items():
    DATASETS[key] = load_and_validate_dataset(key, path)



def inspect_dataset_sample(dataset_key: str, sample_size: int = 1) -> None:
    dataset = DATASETS.get(dataset_key)
    logger.info(
        "Inspecting dataset %s (sample size=%d)",
        dataset_key,
        sample_size
    )

    for instance in dataset[:sample_size]:
        logger.info("ID: %s", instance["id"])
        logger.info("Query: %s", instance["query"])
        logger.info("Answer: %s", instance["answer"])
        logger.info("Positive passages: %d", len(instance["positive"]))
        logger.info("Negative passages: %d", len(instance["negative"]))

        if dataset_key == "en_fact":
            logger.info(
                "Counterfactual passages: %d",
                len(instance["positive_wrong"])
            )

inspect_dataset_sample("en_refine", 1)

# ------------------------------------------------------------------------------
# 1.7 Dataset Statistics
# ------------------------------------------------------------------------------

def summarize_dataset(dataset_key: str) -> Dict[str, Any]:
    dataset = DATASETS[dataset_key]
    total = len(dataset)

    avg_pos = sum(len(i["positive"]) for i in dataset) / total
    avg_neg = sum(len(i["negative"]) for i in dataset) / total

    summary = {
        "dataset": dataset_key,
        "total_instances": total,
        "avg_positive_docs": round(avg_pos, 2),
        "avg_negative_docs": round(avg_neg, 2),
    }

    if dataset_key == "en_fact":
        avg_wrong = sum(len(i["positive_wrong"]) for i in dataset) / total
        summary["avg_counterfactual_docs"] = round(avg_wrong, 2)

    logger.info("Dataset summary: %s", summary)
    return summary

DATASET_SUMMARIES = {
    key: summarize_dataset(key)
    for key in DATASETS.keys()
}

2026-01-24 21:53:49 | INFO | Dataset paths registered


INFO:cap_25_rgbeval:Dataset paths registered


2026-01-24 21:53:49 | INFO | Found dataset file: /content/drive/MyDrive/data/en_refine.json


INFO:cap_25_rgbeval:Found dataset file: /content/drive/MyDrive/data/en_refine.json


2026-01-24 21:53:49 | INFO | Found dataset file: /content/drive/MyDrive/data/en_int.json


...

INFO:cap_25_rgbeval:Dataset summary: {'dataset': 'en_refine', 'total_instances': 300, 'avg_positive_docs': 8.38, 'avg_negative_docs': 18.21}


2026-01-24 21:53:49 | INFO | Dataset summary: {'dataset': 'en_int', 'total_instances': 100, 'avg_positive_docs': 2.05, 'avg_negative_docs': 39.34}


INFO:cap_25_rgbeval:Dataset summary: {'dataset': 'en_int', 'total_instances': 100, 'avg_positive_docs': 2.05, 'avg_negative_docs': 39.34}


2026-01-24 21:53:49 | INFO | Dataset summary: {'dataset': 'en_fact', 'total_instances': 100, 'avg_positive_docs': 3.95, 'avg_negative_docs': 5.94, 'avg_counterfactual_docs': 3.95}


INFO:cap_25_rgbeval:Dataset summary: {'dataset': 'en_fact', 'total_instances': 100, 'avg_positive_docs': 3.95, 'avg_negative_docs': 5.94, 'avg_counterfactual_docs': 3.95}


In [ ]:
# ==============================================================================
# SECTION 2: Prompting and Groq LLM Inference Layer
# ==============================================================================

import os
import time
import json
from typing import Dict, List, Any
from groq import Groq

# ------------------------------------------------------------------------------
# 2.1 Prompt Definition
# ------------------------------------------------------------------------------

RGB_PROMPT_EN = {
    "system": (
        "You are an accurate and reliable AI assistant that can answer questions "
        "with the help of external documents. Please note that external documents "
        "may contain noisy or factually incorrect information. If the information "
        "in the document contains the correct answer, you will give an accurate "
        "answer. If the information in the document does not contain the answer, "
        "you will generate 'I can not answer the question because of the insufficient "
        "information in documents.'. If there are inconsistencies with the facts "
        "in some of the documents, please generate the response "
        "'There are factual errors in the provided documents.' and provide the "
        "correct answer."
    ),
    "instruction": "Document:\n{DOCS}\n\nQuestion:\n{QUERY}"
}

logger.info("RGB instruction.yaml prompt loaded (EN)")

# ------------------------------------------------------------------------------
# 2.2 Prompt Formatter
# ------------------------------------------------------------------------------

def format_rgb_prompt(query: str, documents: List[str]) -> Dict[str, str]:
    docs_text = "\n\n".join(documents)

    user_prompt = RGB_PROMPT_EN["instruction"].format(
        DOCS=docs_text,
        QUERY=query
    )

    return {
        "system": RGB_PROMPT_EN["system"],
        "user": user_prompt
    }

# ------------------------------------------------------------------------------
# 2.3 Groq LLM Client Wrapper (Model-Agnostic)
# ------------------------------------------------------------------------------

class GroqLLMClient:


    def __init__(
        self,
        api_key: str,
        model_name: str,
        temperature: float,
        max_retries: int,
        retry_backoff_sec: int
    ):
        self.client = Groq(api_key=api_key)
        self.model_name = model_name
        self.temperature = temperature
        self.max_retries = max_retries
        self.retry_backoff_sec = retry_backoff_sec

        logger.info(
            "Groq client initialized | model=%s | temperature=%.2f",
            model_name,
            temperature
        )

    def generate(self, prompt: Dict[str, str]) -> str:

        for attempt in range(1, self.max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_name,
                    messages=[
                        {"role": "system", "content": prompt["system"]},
                        {"role": "user", "content": prompt["user"]}
                    ],
                    temperature=self.temperature
                )

                return response.choices[0].message.content.strip()

            except Exception as e:
                logger.warning(
                    "Groq inference failed (attempt %d/%d): %s",
                    attempt,
                    self.max_retries,
                    str(e)
                )

                if attempt == self.max_retries:
                    raise e

                time.sleep(self.retry_backoff_sec)

        raise RuntimeError("Unexpected Groq inference failure")

# ------------------------------------------------------------------------------
# 2.4  Result Loader
# ------------------------------------------------------------------------------

def load_existing_results(result_file: str) -> Dict[str, Any]:
    processed = {}

    if not os.path.exists(result_file):
        return processed

    with open(result_file, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            processed[record["id"]] = record

    logger.info(
        "Loaded %d existing records from %s",
        len(processed),
        result_file
    )
    return processed

def append_result(result_file: str, record: Dict[str, Any]) -> None:
    with open(result_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

2026-01-24 21:53:55 | INFO | RGB instruction.yaml prompt loaded (EN)


INFO:cap_25_rgbeval:RGB instruction.yaml prompt loaded (EN)


In [ ]:
import random

def sample_noise_docs(dataset, current_id, k):
    """
    Sample k irrelevant documents from other dataset instances.
    """
    pool = []
    for inst in dataset:
        if inst.get("id") != current_id:
            pool.extend(inst.get("documents", []))

    if len(pool) == 0:
        return []

    return random.sample(pool, min(k, len(pool)))


In [ ]:
"""
# ==============================================================================
# NEGATIVE REJECTION EVALUATION
# ==============================================================================

import os
import json
import time
from typing import Dict, List, Any
from tqdm import tqdm

# ------------------------------------------------------------------------------
# Negative Rejection: Core Logic
# ------------------------------------------------------------------------------

def evaluate_negative_rejection(
    groq_client,
    dataset: List[Dict[str, Any]],
    model_name: str,
    result_file: str,
    checkpoint_interval: int = 10
) -> Dict[str, float]:
    """
    Evaluate negative rejection capability.

    For negative rejection, we use ONLY negative documents (no positive docs).
    The model should recognize that the documents don't contain the answer
    and reject answering.

    Args:
        groq_client: Initialized Groq LLM client
        dataset: The en_refine.json dataset
        model_name: Name of the model being evaluated
        result_file: Path to save results
        checkpoint_interval: How often to save checkpoints

    Returns:
        Dictionary with rejection rate
    """

    logger.info(f"Starting Negative Rejection evaluation for {model_name}")
    logger.info(f"Total instances: {len(dataset)}")

    # Load existing results if any
    processed_results = load_existing_results(result_file)

    total_instances = 0
    rejection_count = 0

    for idx, instance in enumerate(tqdm(dataset, desc="Negative Rejection")):
        instance_id = instance["id"]

        # Skip if already processed
        if str(instance_id) in processed_results:
            result = processed_results[str(instance_id)]
            total_instances += 1
            if result.get("rejected", False):
                rejection_count += 1
            continue

        query = instance["query"]
        ground_truth = instance["answer"]

        # For negative rejection: Use ONLY negative documents
        # The model should NOT be able to answer from these
        negative_docs = instance["negative"]

        # Take first 5 negative documents (as per RGB paper setup)
        selected_docs = negative_docs[:5]

        # Format prompt
        prompt = format_rgb_prompt(query, selected_docs)

        # Generate response
        try:
            response = groq_client.generate(prompt)

            # Check if response indicates rejection
            is_rejected = check_rejection(response)

            # Save result
            result_record = {
                "id": instance_id,
                "query": query,
                "ground_truth": ground_truth,
                "response": response,
                "rejected": is_rejected,
                "model": model_name,
                "num_docs": len(selected_docs)
            }

            append_result(result_file, result_record)

            total_instances += 1
            if is_rejected:
                rejection_count += 1

            # Checkpoint logging
            if (idx + 1) % checkpoint_interval == 0:
                current_rate = (rejection_count / total_instances) * 100
                logger.info(
                    f"Checkpoint {idx + 1}/{len(dataset)} | "
                    f"Rejection Rate: {current_rate:.2f}%"
                )

        except Exception as e:
            logger.error(f"Error processing instance {instance_id}: {str(e)}")
            continue

    # Calculate final metrics
    rejection_rate = (rejection_count / total_instances) * 100 if total_instances > 0 else 0.0

    metrics = {
        "rejection_rate": rejection_rate,
        "total_rejected": rejection_count,
        "total_instances": total_instances,
        "model": model_name
    }

    logger.info(f"Negative Rejection Results for {model_name}:")
    logger.info(f"  Rejection Rate: {rejection_rate:.2f}%")
    logger.info(f"  Total Rejected: {rejection_count}/{total_instances}")

    return metrics


# ------------------------------------------------------------------------------
# Rejection Detection Logic
# ------------------------------------------------------------------------------

def check_rejection(response: str) -> bool:
    """
    Check if the model's response indicates rejection.

    According to RGB paper, rejection phrases include:
    - "I can not answer the question"
    - "insufficient information"
    - "cannot be answered"
    - Similar variations

    Args:
        response: The model's generated response

    Returns:
        True if response indicates rejection, False otherwise
    """
    response_lower = response.lower()

    # Define rejection indicators based on RGB paper prompt
    rejection_indicators = [
        "i can not answer",
        "i cannot answer",
        "can not answer the question",
        "cannot answer the question",
        "insufficient information",
        "not enough information",
        "do not contain",
        "does not contain",
        "don't contain",
        "doesn't contain",
        "unable to answer",
        "no information",
        "lack of information",
        "cannot be answered",
        "can not be answered",
        "not able to answer",
        "documents do not provide",
        "documents don't provide",
    ]

    # Check if any rejection indicator is present
    for indicator in rejection_indicators:
        if indicator in response_lower:
            return True

    return False


# ------------------------------------------------------------------------------
# Run Negative Rejection Evaluation
# ------------------------------------------------------------------------------

def run_negative_rejection_experiment(
    api_key: str,
    model_configs: List[Dict[str, str]],
    dataset_path: str,
    results_dir: str
):
    """
    Run negative rejection evaluation for multiple models.

    Args:
        api_key: Groq API key
        model_configs: List of dicts with 'name' and 'm
"""

In [ ]:
# ==============================================================================
# NEGATIVE REJECTION EVALUATION
# ==============================================================================

import os
import json
import time
from typing import Dict, List, Any
from tqdm import tqdm

# ------------------------------------------------------------------------------
# Negative Rejection: Core Logic
# ------------------------------------------------------------------------------

def evaluate_negative_rejection(
    groq_client,
    dataset: List[Dict[str, Any]],
    model_name: str,
    result_file: str,
    checkpoint_interval: int = 10
) -> Dict[str, float]:
    """
    Evaluate negative rejection capability.

    For negative rejection, we use ONLY negative documents (no positive docs).
    The model should recognize that the documents don't contain the answer
    and reject answering.

    Args:
        groq_client: Initialized Groq LLM client
        dataset: The en_refine.json dataset
        model_name: Name of the model being evaluated
        result_file: Path to save results
        checkpoint_interval: How often to save checkpoints

    Returns:
        Dictionary with rejection rate
    """

    logger.info(f"Starting Negative Rejection evaluation for {model_name}")
    logger.info(f"Total instances: {len(dataset)}")

    # Load existing results if any
    processed_results = load_existing_results(result_file)

    total_instances = 0
    rejection_count = 0

    for idx, instance in enumerate(tqdm(dataset, desc="Negative Rejection")):
        instance_id = instance["id"]

        # Skip if already processed
        if str(instance_id) in processed_results:
            result = processed_results[str(instance_id)]
            total_instances += 1
            if result.get("rejected", False):
                rejection_count += 1
            continue

        query = instance["query"]
        ground_truth = instance["answer"]

        # For negative rejection: Use ONLY negative documents
        # The model should NOT be able to answer from these
        negative_docs = instance["negative"]

        # Take first 5 negative documents (as per RGB paper setup)
        selected_docs = negative_docs[:5]

        # Format prompt
        prompt = format_rgb_prompt(query, selected_docs)

        # Generate response
        try:
            response = groq_client.generate(prompt)

            # Check if response indicates rejection
            is_rejected = check_rejection(response)

            # Save result
            result_record = {
                "id": instance_id,
                "query": query,
                "ground_truth": ground_truth,
                "response": response,
                "rejected": is_rejected,
                "model": model_name,
                "num_docs": len(selected_docs)
            }

            append_result(result_file, result_record)

            total_instances += 1
            if is_rejected:
                rejection_count += 1

            # Checkpoint logging
            if (idx + 1) % checkpoint_interval == 0:
                current_rate = (rejection_count / total_instances) * 100
                logger.info(
                    f"Checkpoint {idx + 1}/{len(dataset)} | "
                    f"Rejection Rate: {current_rate:.2f}%"
                )

        except Exception as e:
            logger.error(f"Error processing instance {instance_id}: {str(e)}")
            continue

    # Calculate final metrics
    rejection_rate = (rejection_count / total_instances) * 100 if total_instances > 0 else 0.0

    metrics = {
        "rejection_rate": rejection_rate,
        "total_rejected": rejection_count,
        "total_instances": total_instances,
        "model": model_name
    }

    logger.info(f"Negative Rejection Results for {model_name}:")
    logger.info(f"  Rejection Rate: {rejection_rate:.2f}%")
    logger.info(f"  Total Rejected: {rejection_count}/{total_instances}")

    return metrics


# ------------------------------------------------------------------------------
# Rejection Detection Logic
# ------------------------------------------------------------------------------

def check_rejection(response: str) -> bool:
    """
    Check if the model's response indicates rejection.

    According to RGB paper, rejection phrases include:
    - "I can not answer the question"
    - "insufficient information"
    - "cannot be answered"
    - Similar variations

    Args:
        response: The model's generated response

    Returns:
        True if response indicates rejection, False otherwise
    """
    response_lower = response.lower()

    # Define rejection indicators based on RGB paper prompt
    rejection_indicators = [
        "i can not answer",
        "i cannot answer",
        "can not answer the question",
        "cannot answer the question",
        "insufficient information",
        "not enough information",
        "do not contain",
        "does not contain",
        "don't contain",
        "doesn't contain",
        "unable to answer",
        "no information",
        "lack of information",
        "cannot be answered",
        "can not be answered",
        "not able to answer",
        "documents do not provide",
        "documents don't provide",
    ]

    # Check if any rejection indicator is present
    for indicator in rejection_indicators:
        if indicator in response_lower:
            return True

    return False


# ------------------------------------------------------------------------------
# Run Negative Rejection Evaluation
# ------------------------------------------------------------------------------

def run_negative_rejection_experiment(
    api_key: str,
    model_configs: List[Dict[str, str]],
    dataset_path: str,
    results_dir: str
):
    """
    Run negative rejection evaluation for multiple models.

    Args:
        api_key: Groq API key
        model_configs: List of dicts with 'name' and 'model_id' keys
        dataset_path: Path to en_refine.json
        results_dir: Directory to save results
    """

    # Load dataset
    dataset = load_jsonl(dataset_path)
    logger.info(f"Loaded {len(dataset)} instances from {dataset_path}")

    all_results = {}

    for model_config in model_configs:
        model_short_name = model_config["name"]
        model_id = model_config["model_id"]

        logger.info(f"\n{'='*80}")
        logger.info(f"Evaluating model: {model_short_name} ({model_id})")
        logger.info(f"{'='*80}\n")

        # Initialize Groq client for this model
        groq_client = GroqLLMClient(
            api_key=api_key,
            model_name=model_id,
            temperature=0.0,
            max_retries=3,
            retry_backoff_sec=5
        )

        # Create result file path
        result_file = os.path.join(
            results_dir,
            f"negative_rejection_{model_short_name}.jsonl"
        )

        # Run evaluation
        metrics = evaluate_negative_rejection(
            groq_client=groq_client,
            dataset=dataset,
            model_name=model_short_name,
            result_file=result_file
        )

        all_results[model_short_name] = metrics

    # Save summary
    summary_file = os.path.join(results_dir, "negative_rejection_summary.json")
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    logger.info(f"\n{'='*80}")
    logger.info("NEGATIVE REJECTION - SUMMARY")
    logger.info(f"{'='*80}\n")

    print("\n| Model | Rejection Rate (%) |")
    print("|-------|-------------------|")
    for model_name, metrics in all_results.items():
        print(f"| {model_name} | {metrics['rejection_rate']:.2f} |")

    return all_results


# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

# Define your models (at least 3 as per assignment requirement)
MODEL_CONFIGS = [
    {
        "name": "qwen3_32b",
        "model_id": "qwen/qwen3-32b"
    },
    {
        "name": "llama3_1_8b",
        "model_id": "llama-3.1-8b-instant"
    },
    {
        "name": "llama3_3_70b",
        "model_id": "llama-3.3-70b-versatile"
    }
]



# Paths
DATASET_PATH = os.path.join(DATASET_DIR, "en_refine.json")
RESULTS_DIR_NEG_REJ = os.path.join(RESULTS_DIR, "negative_rejection")
os.makedirs(RESULTS_DIR_NEG_REJ, exist_ok=True)

# Run experiment
results = run_negative_rejection_experiment(
    api_key=GROQ_API_KEY,
    model_configs=MODEL_CONFIGS,
    dataset_path=DATASET_PATH,
    results_dir=RESULTS_DIR_NEG_REJ
)


2026-01-17 10:59:24 | INFO | Loaded 300 instances from /content/drive/MyDrive/data/en_refine.json


INFO:cap_25_rgbeval:Loaded 300 instances from /content/drive/MyDrive/data/en_refine.json


2026-01-17 10:59:24 | INFO | 


INFO:cap_25_rgbeval:


2026-01-17 10:59:24 | INFO | Evaluating model: qwen3_32b (qwen/qwen3-32b)


...

ERROR:cap_25_rgbeval:Error processing instance 47: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdf5f4a2f3krsf5ss687c62a` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99690, Requested 1104. Please try again in 11m26.016s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Negative Rejection:  16%|█▌        | 48/300 [05:36<42:47, 10.19s/it]

2026-01-17 13:18:45 | WARNING | Groq inference failed (attempt 1/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdf5f4a2f3krsf5ss687c62a` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99690, Requested 1072. Please try again in 10m58.368s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


2026-01-17 13:18:50 | WARNING | Groq inference failed (attempt 2/3): Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdf5f4a2f3krsf5ss687c62a` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99685, Requested 1072. Please try again in 10m54.048s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


Negative Rejection:  16%|█▌        | 48/300 [05:46<30:17,  7.21s/it]


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# LOAD AND ANALYZE NEGATIVE REJECTION RESULTS
# ==============================================================================

import os
import json
import pandas as pd
from typing import Dict, List, Any

# ------------------------------------------------------------------------------
# Load Results from JSONL Files
# ------------------------------------------------------------------------------

def load_negative_rejection_results(results_dir: str) -> Dict[str, List[Dict]]:
    """
    Load all negative rejection JSONL result files from the results directory.

    Args:
        results_dir: Directory containing the negative rejection results

    Returns:
        Dictionary mapping model names to their results
    """
    all_results = {}

    # List all JSONL files in the directory
    if not os.path.exists(results_dir):
        print(f"Error: Directory {results_dir} does not exist")
        return all_results

    for filename in os.listdir(results_dir):
        if filename.startswith("negative_rejection_") and filename.endswith(".jsonl"):
            filepath = os.path.join(results_dir, filename)

            # Extract model name from filename
            # Format: negative_rejection_{model_name}.jsonl
            model_name = filename.replace("negative_rejection_", "").replace(".jsonl", "")

            # Load JSONL file
            results = []
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        results.append(json.loads(line))

            all_results[model_name] = results
            print(f"✓ Loaded {len(results)} results for {model_name}")

    return all_results


# ------------------------------------------------------------------------------
# Calculate Metrics for Each Model
# ------------------------------------------------------------------------------

def calculate_rejection_metrics(results: List[Dict]) -> Dict[str, Any]:
    """
    Calculate rejection rate and other metrics from results.

    Args:
        results: List of result dictionaries

    Returns:
        Dictionary with calculated metrics
    """
    total_instances = len(results)
    rejection_count = sum(1 for r in results if r.get("rejected", False))

    rejection_rate = (rejection_count / total_instances * 100) if total_instances > 0 else 0.0

    return {
        "total_instances": total_instances,
        "total_rejected": rejection_count,
        "total_not_rejected": total_instances - rejection_count,
        "rejection_rate": round(rejection_rate, 2)
    }


# ------------------------------------------------------------------------------
# Generate Summary Table
# ------------------------------------------------------------------------------

def generate_summary_table(all_results: Dict[str, List[Dict]]) -> pd.DataFrame:
    """
    Generate a summary table with rejection rates for all models.

    Args:
        all_results: Dictionary mapping model names to their results

    Returns:
        DataFrame with summary statistics
    """
    summary_data = []

    for model_name, results in all_results.items():
        metrics = calculate_rejection_metrics(results)
        summary_data.append({
            "Model": model_name,
            "Total Instances": metrics["total_instances"],
            "Rejected": metrics["total_rejected"],
            "Not Rejected": metrics["total_not_rejected"],
            "Rejection Rate (%)": metrics["rejection_rate"]
        })

    # Create DataFrame and sort by rejection rate (descending)
    df = pd.DataFrame(summary_data)
    df = df.sort_values("Rejection Rate (%)", ascending=False)

    return df


# ------------------------------------------------------------------------------
# Export Functions
# ------------------------------------------------------------------------------

def export_summary_to_csv(df: pd.DataFrame, output_file: str):
    """Export summary table to CSV."""
    df.to_csv(output_file, index=False)
    print(f"✓ Summary exported to {output_file}")


def export_summary_to_markdown(df: pd.DataFrame, output_file: str):
    """Export summary table to Markdown format."""
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("# Negative Rejection Results\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n")
    print(f"✓ Markdown summary exported to {output_file}")


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

# Set your results directory path
RESULTS_DIR_NEG_REJ = os.path.join(RESULTS_DIR, "negative_rejection")

# Load all results
print("Loading negative rejection results...\n")
all_model_results = load_negative_rejection_results(RESULTS_DIR_NEG_REJ)

if len(all_model_results) == 0:
    print("No results found! Check your results directory path.")
else:
    print(f"\nFound results for {len(all_model_results)} models\n")

    # Generate summary table
    summary_df = generate_summary_table(all_model_results)

    # Display the table
    print("="*80)
    print("NEGATIVE REJECTION SUMMARY TABLE")
    print("="*80)
    print()
    print(summary_df.to_string(index=False))
    print()

    # Display in markdown format (for your report)
    print("="*80)
    print("MARKDOWN FORMAT (Copy this for your report)")
    print("="*80)
    print()
    print(summary_df.to_markdown(index=False))
    print()

    # Save to files
    summary_csv = os.path.join(RESULTS_DIR_NEG_REJ, "summary_table.csv")
    summary_md = os.path.join(RESULTS_DIR_NEG_REJ, "summary_table.md")

    export_summary_to_csv(summary_df, summary_csv)
    export_summary_to_markdown(summary_df, summary_md)

    # Detailed breakdown
    print("="*80)
    print("DETAILED BREAKDOWN BY MODEL")
    print("="*80)
    print()

    for model_name, results in all_model_results.items():
        total = len(results)
        rejected = sum(1 for r in results if r.get("rejected", False))
        not_rejected = total - rejected

        print(f"Model: {model_name}")
        print(f"  - Total: {total}")
        print(f"  - Rejected: {rejected} ({rejected/total*100:.2f}%)")
        print(f"  - Not Rejected: {not_rejected} ({not_rejected/total*100:.2f}%)")
        print()


Loading negative rejection results...

✓ Loaded 300 results for qwen3_32b
✓ Loaded 300 results for llama3_1_8b
✓ Loaded 23 results for llama3_3_70b

Found results for 3 models

NEGATIVE REJECTION SUMMARY TABLE

       Model  Total Instances  Rejected  Not Rejected  Rejection Rate (%)
llama3_3_70b               23        20             3               86.96
   qwen3_32b              300       234            66               78.00
 llama3_1_8b              300       222            78               74.00

MARKDOWN FORMAT (Copy this for your report)

| Model        |   Total Instances |   Rejected |   Not Rejected |   Rejection Rate (%) |
|:-------------|------------------:|-----------:|---------------:|---------------------:|
| llama3_3_70b |                23 |         20 |              3 |                86.96 |
| qwen3_32b    |               300 |        234 |             66 |                78    |
| llama3_1_8b  |               300 |        222 |             78 |                74   

In [ ]:
##information integration


# ==============================================================================
# INFORMATION INTEGRATION EVALUATION
# ==============================================================================

import os
import json
import random
from typing import Dict, List, Any, Tuple
from tqdm import tqdm

# ------------------------------------------------------------------------------
# Information Integration: Document Selection Logic
# ------------------------------------------------------------------------------

def select_documents_for_information_integration(
    instance: Dict[str, Any],
    noise_ratio: float,
    passage_num: int = 5
) -> Tuple[List[str], str]:
    """
    Select documents for information integration task.

    Key differences from negative rejection:
    - Uses POSITIVE documents (which contain parts of the answer)
    - Positive documents are grouped (each group answers one sub-question)
    - Need to integrate information from multiple positive document groups
    - Can add negative documents based on noise ratio

    Args:
        instance: Data instance from en_int.json
        noise_ratio: Ratio of noise documents (0.0, 0.2, 0.4)
        passage_num: Total number of documents to provide (default: 5)

    Returns:
        Tuple of (selected_documents, document_composition_info)
    """
    # In en_int.json, positive is a LIST OF LISTS
    # Each inner list contains documents that answer one sub-question
    positive_groups = instance["positive"]  # e.g., [[doc1, doc2], [doc3, doc4]]
    negative_docs = instance["negative"]

    # Calculate how many documents to use
    num_noise = int(passage_num * noise_ratio)
    num_positive = passage_num - num_noise

    selected_docs = []

    # Select positive documents from DIFFERENT groups
    # This ensures we need to integrate information from multiple sources
    if len(positive_groups) > 0:
        # Distribute positive slots across groups
        docs_per_group = max(1, num_positive // len(positive_groups))

        for group in positive_groups:
            if len(selected_docs) < num_positive and len(group) > 0:
                # Sample from this group
                sample_count = min(docs_per_group, len(group), num_positive - len(selected_docs))
                selected_docs.extend(random.sample(group, sample_count))

    # Fill remaining slots with negative documents (noise)
    if num_noise > 0 and len(negative_docs) > 0:
        sample_count = min(num_noise, len(negative_docs))
        selected_docs.extend(random.sample(negative_docs, sample_count))

    # Shuffle to randomize order
    random.shuffle(selected_docs)

    # Create composition info for logging
    composition = f"Positive={num_positive}, Noise={num_noise}, Total={len(selected_docs)}"

    return selected_docs, composition


# ------------------------------------------------------------------------------
# Answer Checking for Information Integration
# ------------------------------------------------------------------------------

def check_information_integration_answer(response: str, ground_truth: List[List[str]]) -> bool:
    """
    Check if response contains ALL required answers for information integration.

    For information integration, the answer is a list of lists.
    Each inner list represents acceptable answers for one sub-question.
    The response must contain at least one answer from EACH sub-question.

    Args:
        response: Model's generated response
        ground_truth: List of answer groups, e.g.,
                     [["May 18", "May 18th"], ["March 1", "March 1st"]]

    Returns:
        True if ALL sub-questions are answered correctly
    """
    response_lower = response.lower()

    # Check if at least one answer from each group is present
    all_answered = True

    for answer_group in ground_truth:
        # Check if ANY answer variant from this group appears in response
        group_answered = False
        for answer_variant in answer_group:
            if answer_variant.lower() in response_lower:
                group_answered = True
                break

        if not group_answered:
            all_answered = False
            break

    return all_answered


# ------------------------------------------------------------------------------
# Information Integration Evaluation
# ------------------------------------------------------------------------------

def evaluate_information_integration(
    groq_client,
    dataset: List[Dict[str, Any]],
    model_name: str,
    noise_ratio: float,
    result_file: str,
    checkpoint_interval: int = 10
) -> Dict[str, Any]:
    """
    Evaluate information integration capability at a specific noise ratio.

    Args:
        groq_client: Initialized Groq LLM client
        dataset: The en_int.json dataset
        model_name: Name of the model being evaluated
        noise_ratio: Noise ratio (0.0, 0.2, or 0.4)
        result_file: Path to save results
        checkpoint_interval: How often to save checkpoints

    Returns:
        Dictionary with accuracy metrics
    """

    logger.info(f"Starting Information Integration evaluation for {model_name} at noise_ratio={noise_ratio}")
    logger.info(f"Total instances: {len(dataset)}")

    # Load existing results if any
    processed_results = load_existing_results(result_file)

    total_instances = 0
    correct_count = 0

    for idx, instance in enumerate(tqdm(dataset, desc=f"InfoInteg (noise={noise_ratio})")):
        instance_id = instance["id"]

        # Skip if already processed
        if str(instance_id) in processed_results:
            result = processed_results[str(instance_id)]
            total_instances += 1
            if result.get("correct", False):
                correct_count += 1
            continue

        query = instance["query"]
        ground_truth = instance["answer"]  # List of lists

        # Select documents for this noise ratio
        selected_docs, composition = select_documents_for_information_integration(
            instance, noise_ratio, passage_num=5
        )

        # Format prompt
        prompt = format_rgb_prompt(query, selected_docs)

        # Generate response
        try:
            response = groq_client.generate(prompt)

            # Check if answer is correct (must integrate all sub-answers)
            is_correct = check_information_integration_answer(response, ground_truth)

            # Save result
            result_record = {
                "id": instance_id,
                "query": query,
                "ground_truth": ground_truth,
                "response": response,
                "correct": is_correct,
                "noise_ratio": noise_ratio,
                "doc_composition": composition,
                "model": model_name,
                "num_docs": len(selected_docs)
            }

            append_result(result_file, result_record)

            total_instances += 1
            if is_correct:
                correct_count += 1

            # Checkpoint logging
            if (idx + 1) % checkpoint_interval == 0:
                current_acc = (correct_count / total_instances) * 100
                logger.info(
                    f"Checkpoint {idx + 1}/{len(dataset)} | "
                    f"Accuracy: {current_acc:.2f}%"
                )

        except Exception as e:
            logger.error(f"Error processing instance {instance_id}: {str(e)}")
            continue

    # Calculate final metrics
    accuracy = (correct_count / total_instances) * 100 if total_instances > 0 else 0.0

    metrics = {
        "accuracy": accuracy,
        "total_correct": correct_count,
        "total_instances": total_instances,
        "noise_ratio": noise_ratio,
        "model": model_name
    }

    logger.info(f"Information Integration Results for {model_name} (noise={noise_ratio}):")
    logger.info(f"  Accuracy: {accuracy:.2f}%")
    logger.info(f"  Correct: {correct_count}/{total_instances}")

    return metrics


# ------------------------------------------------------------------------------
# Run Information Integration Experiment (All Noise Ratios)
# ------------------------------------------------------------------------------

def run_information_integration_experiment(
    api_key: str,
    model_configs: List[Dict[str, str]],
    dataset_path: str,
    results_dir: str,
    noise_ratios: List[float] = [0.0, 0.2, 0.4]
):
    """
    Run information integration evaluation for multiple models and noise ratios.

    Args:
        api_key: Groq API key
        model_configs: List of dicts with 'name' and 'model_id' keys
        dataset_path: Path to en_int.json
        results_dir: Directory to save results
        noise_ratios: List of noise ratios to test (default: [0.0, 0.2, 0.4])
    """

    # Load dataset
    dataset = load_jsonl(dataset_path)
    logger.info(f"Loaded {len(dataset)} instances from {dataset_path}")

    all_results = {}

    for model_config in model_configs:
        model_short_name = model_config["name"]
        model_id = model_config["model_id"]

        logger.info(f"\n{'='*80}")
        logger.info(f"Evaluating model: {model_short_name} ({model_id})")
        logger.info(f"{'='*80}\n")

        # Initialize Groq client for this model
        groq_client = GroqLLMClient(
            api_key=api_key,
            model_name=model_id,
            temperature=0.0,
            max_retries=3,
            retry_backoff_sec=5
        )

        model_results = {}

        # Evaluate at each noise ratio
        for noise_ratio in noise_ratios:
            logger.info(f"\n--- Noise Ratio: {noise_ratio} ---")

            # Create result file path
            result_file = os.path.join(
                results_dir,
                f"information_integration_{model_short_name}_noise_{noise_ratio}.jsonl"
            )

            # Run evaluation for this noise ratio
            metrics = evaluate_information_integration(
                groq_client=groq_client,
                dataset=dataset,
                model_name=model_short_name,
                noise_ratio=noise_ratio,
                result_file=result_file
            )

            model_results[f"noise_{noise_ratio}"] = metrics

        all_results[model_short_name] = model_results

    # Save summary
    summary_file = os.path.join(results_dir, "information_integration_summary.json")
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    # Print summary table
    logger.info(f"\n{'='*80}")
    logger.info("INFORMATION INTEGRATION - SUMMARY TABLE")
    logger.info(f"{'='*80}\n")

    print("\n| Model | Noise 0.0 | Noise 0.2 | Noise 0.4 |")
    print("|-------|-----------|-----------|-----------|")
    for model_name, results in all_results.items():
        acc_0 = results.get("noise_0.0", {}).get("accuracy", 0)
        acc_2 = results.get("noise_0.2", {}).get("accuracy", 0)
        acc_4 = results.get("noise_0.4", {}).get("accuracy", 0)
        print(f"| {model_name} | {acc_0:.2f} | {acc_2:.2f} | {acc_4:.2f} |")

    return all_results


# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

# Define your models
MODEL_CONFIGS = [
    {"name": "llama3_1_8b", "model_id": "llama-3.1-8b-instant"}
]


# Paths
DATASET_PATH_INT = os.path.join(DATASET_DIR, "en_int.json")
RESULTS_DIR_INFO_INTEG = os.path.join(RESULTS_DIR, "information_integration")
os.makedirs(RESULTS_DIR_INFO_INTEG, exist_ok=True)

# Run experiment with noise ratios 0.0, 0.2, 0.4
results = run_information_integration_experiment(
    api_key=GROQ_API_KEY,
    model_configs=MODEL_CONFIGS,
    dataset_path=DATASET_PATH_INT,
    results_dir=RESULTS_DIR_INFO_INTEG,
    noise_ratios=[0.2, 0.4]
)


2026-01-18 03:04:46 | INFO | Loaded 100 instances from /content/drive/MyDrive/data/en_int.json


INFO:cap_25_rgbeval:Loaded 100 instances from /content/drive/MyDrive/data/en_int.json


2026-01-18 03:04:46 | INFO | 


INFO:cap_25_rgbeval:


2026-01-18 03:04:46 | INFO | Evaluating model: llama3_1_8b (llama-3.1-8b-instant)


...

2026-01-18 03:40:38 | INFO | INFORMATION INTEGRATION - SUMMARY TABLE


INFO:cap_25_rgbeval:INFORMATION INTEGRATION - SUMMARY TABLE


2026-01-18 03:40:39 | INFO | ================================================================================



INFO:cap_25_rgbeval:================================================================================




| Model | Noise 0.0 | Noise 0.2 | Noise 0.4 |
|-------|-----------|-----------|-----------|
| llama3_1_8b | 0.00 | 95.00 | 98.00 |


In [ ]:
# ==============================================================================
# INFORMATION INTEGRATION RESULTS ANALYSIS AND REPORTING
# ==============================================================================

import os
import json
import pandas as pd
from typing import Dict, List, Any

# ------------------------------------------------------------------------------
# Load Information Integration Results
# ------------------------------------------------------------------------------

def load_information_integration_results(results_dir: str) -> Dict[str, Dict[str, List[Dict]]]:
    """
    Load all information integration JSONL result files.

    Returns:
        Nested dict: {model_name: {noise_ratio: [results]}}
    """
    all_results = {}

    if not os.path.exists(results_dir):
        print(f"Error: Directory {results_dir} does not exist")
        return all_results

    for filename in os.listdir(results_dir):
        if filename.startswith("information_integration_") and filename.endswith(".jsonl"):
            filepath = os.path.join(results_dir, filename)

            # Parse filename: information_integration_{model_name}_noise_{noise_ratio}.jsonl
            parts = filename.replace("information_integration_", "").replace(".jsonl", "").split("_noise_")

            if len(parts) == 2:
                model_name = parts[0]
                noise_ratio = parts[1]

                # Load JSONL file
                results = []
                with open(filepath, "r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            results.append(json.loads(line))

                # Initialize model dict if needed
                if model_name not in all_results:
                    all_results[model_name] = {}

                all_results[model_name][f"noise_{noise_ratio}"] = results
                print(f"✓ Loaded {len(results)} results for {model_name} at noise={noise_ratio}")

    return all_results


# ------------------------------------------------------------------------------
# Calculate Accuracy Metrics
# ------------------------------------------------------------------------------

def calculate_accuracy_metrics(results: List[Dict]) -> Dict[str, Any]:
    """
    Calculate accuracy metrics from information integration results.
    """
    total_instances = len(results)
    correct_count = sum(1 for r in results if r.get("correct", False))

    accuracy = (correct_count / total_instances * 100) if total_instances > 0 else 0.0

    return {
        "total_instances": total_instances,
        "total_correct": correct_count,
        "total_incorrect": total_instances - correct_count,
        "accuracy": round(accuracy, 2)
    }


# ------------------------------------------------------------------------------
# Generate Summary Table
# ------------------------------------------------------------------------------

def generate_info_integration_summary_table(
    all_results: Dict[str, Dict[str, List[Dict]]]
) -> pd.DataFrame:
    """
    Generate summary table with accuracy across noise ratios.
    """
    summary_data = []

    for model_name, noise_results in all_results.items():
        row_data = {"Model": model_name}

        # Get accuracy for each noise ratio
        for noise_key in ["noise_0.0", "noise_0.2", "noise_0.4"]:
            if noise_key in noise_results:
                metrics = calculate_accuracy_metrics(noise_results[noise_key])
                row_data[f"Noise {noise_key.split('_')[1]}"] = metrics["accuracy"]
            else:
                row_data[f"Noise {noise_key.split('_')[1]}"] = None

        summary_data.append(row_data)

    # Create DataFrame and sort by Noise 0.0 accuracy (descending)
    df = pd.DataFrame(summary_data)
    if "Noise 0.0" in df.columns:
        df = df.sort_values("Noise 0.0", ascending=False)

    return df


# ------------------------------------------------------------------------------
# Detailed Breakdown
# ------------------------------------------------------------------------------

def print_detailed_breakdown(all_results: Dict[str, Dict[str, List[Dict]]]):
    """Print detailed breakdown for each model and noise ratio."""

    print("="*80)
    print("DETAILED BREAKDOWN BY MODEL AND NOISE RATIO")
    print("="*80)
    print()

    for model_name, noise_results in all_results.items():
        print(f"Model: {model_name}")
        print("-" * 60)

        for noise_key in sorted(noise_results.keys()):
            results = noise_results[noise_key]
            metrics = calculate_accuracy_metrics(results)
            noise_value = noise_key.split("_")[1]

            print(f"  Noise Ratio {noise_value}:")
            print(f"    - Total Instances: {metrics['total_instances']}")
            print(f"    - Correct: {metrics['total_correct']}")
            print(f"    - Incorrect: {metrics['total_incorrect']}")
            print(f"    - Accuracy: {metrics['accuracy']:.2f}%")

        print()


# ------------------------------------------------------------------------------
# Compare with RGB Paper Benchmarks
# ------------------------------------------------------------------------------

def compare_with_rgb_paper(summary_df: pd.DataFrame):
    """Compare results with RGB paper benchmarks."""

    print("="*80)
    print("COMPARISON WITH RGB PAPER BENCHMARKS")
    print("="*80)
    print()

    # RGB paper results (Table 5 - English)
    rgb_benchmarks = {
        "ChatGPT": {"0.0": 55, "0.2": 51, "0.4": 34},
        "Qwen-7B-Chat": {"0.0": 55, "0.2": 50, "0.4": 37},
        "Vicuna-7B-v1.3": {"0.0": 60, "0.2": 53, "0.4": 43},
        "ChatGLM-6B": {"0.0": 45, "0.2": 36, "0.4": 35},
        "ChatGLM2-6B": {"0.0": 34, "0.2": 32, "0.4": 21},
        "BELLE-7B-2M": {"0.0": 40, "0.2": 34, "0.4": 24}
    }

    print("RGB Paper Best Results (English):")
    print("| Model | Noise 0.0 | Noise 0.2 | Noise 0.4 |")
    print("|-------|-----------|-----------|-----------|")
    for model, scores in rgb_benchmarks.items():
        print(f"| {model} | {scores['0.0']:.0f} | {scores['0.2']:.0f} | {scores['0.4']:.0f} |")

    print()
    print("Your Results:")
    print(summary_df.to_markdown(index=False))
    print()


# ------------------------------------------------------------------------------
# Export Functions
# ------------------------------------------------------------------------------

def export_summary_to_csv(df: pd.DataFrame, output_file: str):
    """Export summary table to CSV."""
    df.to_csv(output_file, index=False)
    print(f"✓ Summary exported to {output_file}")


def export_summary_to_markdown(df: pd.DataFrame, output_file: str):
    """Export summary table to Markdown format."""
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("# Information Integration Results\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n")
    print(f"✓ Markdown summary exported to {output_file}")


def export_detailed_results(all_results: Dict[str, Dict[str, List[Dict]]], output_dir: str):
    """Export detailed analysis for each model."""

    for model_name, noise_results in all_results.items():
        model_summary = {
            "model": model_name,
            "noise_ratios": {}
        }

        for noise_key, results in noise_results.items():
            metrics = calculate_accuracy_metrics(results)

            # Get correct and incorrect instances
            correct_instances = [r for r in results if r.get("correct", False)]
            incorrect_instances = [r for r in results if not r.get("correct", False)]

            model_summary["noise_ratios"][noise_key] = {
                "metrics": metrics,
                "sample_correct": correct_instances[:5],
                "sample_incorrect": incorrect_instances[:5]
            }

        # Save to file
        output_file = os.path.join(output_dir, f"{model_name}_detailed_analysis.json")
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(model_summary, f, indent=2, ensure_ascii=False)

        print(f"✓ Detailed analysis exported for {model_name}")


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

# Set your results directory path
RESULTS_DIR_INFO_INTEG = os.path.join(RESULTS_DIR, "information_integration")

# Load all results
print("Loading information integration results...\n")
all_model_results = load_information_integration_results(RESULTS_DIR_INFO_INTEG)

if len(all_model_results) == 0:
    print("No results found! Check your results directory path.")
else:
    print(f"\nFound results for {len(all_model_results)} models\n")

    # Generate summary table
    summary_df = generate_info_integration_summary_table(all_model_results)

    # Display the table
    print("="*80)
    print("INFORMATION INTEGRATION SUMMARY TABLE")
    print("="*80)
    print()
    print(summary_df.to_string(index=False))
    print()

    # Display in markdown format (for your report)
    print("="*80)
    print("MARKDOWN FORMAT (Copy this for your report)")
    print("="*80)
    print()
    print(summary_df.to_markdown(index=False))
    print()

    # Save to files
    summary_csv = os.path.join(RESULTS_DIR_INFO_INTEG, "summary_table.csv")
    summary_md = os.path.join(RESULTS_DIR_INFO_INTEG, "summary_table.md")

    export_summary_to_csv(summary_df, summary_csv)
    export_summary_to_markdown(summary_df, summary_md)

    # Detailed breakdown
    print_detailed_breakdown(all_model_results)

    # Compare with RGB paper
    compare_with_rgb_paper(summary_df)

    # Export detailed analysis
    print("="*80)
    print("EXPORTING DETAILED ANALYSIS")
    print("="*80)
    print()
    export_detailed_results(all_model_results, RESULTS_DIR_INFO_INTEG)

    print()
    print("="*80)
    print("REPORT GENERATION COMPLETE")
    print("="*80)


Loading information integration results...

✓ Loaded 100 results for qwen3_32b at noise=0.0
✓ Loaded 100 results for qwen3_32b at noise=0.2
✓ Loaded 100 results for qwen3_32b at noise=0.4
✓ Loaded 82 results for llama3_1_8b at noise=0.0
✓ Loaded 100 results for llama3_1_8b at noise=0.2
✓ Loaded 100 results for llama3_1_8b at noise=0.4

Found results for 2 models

INFORMATION INTEGRATION SUMMARY TABLE

      Model  Noise 0.0  Noise 0.2  Noise 0.4
  qwen3_32b      99.00      100.0       98.0
llama3_1_8b      96.34       95.0       98.0

MARKDOWN FORMAT (Copy this for your report)

| Model       |   Noise 0.0 |   Noise 0.2 |   Noise 0.4 |
|:------------|------------:|------------:|------------:|
| qwen3_32b   |       99    |         100 |          98 |
| llama3_1_8b |       96.34 |          95 |          98 |

✓ Summary exported to /content/drive/MyDrive/results_01/information_integration/summary_table.csv
✓ Markdown summary exported to /content/drive/MyDrive/results_01/information_integr

In [ ]:
from google.colab import userdata
import os

# Retrieve the API key from Colab secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("✓ API key loaded from Colab secrets")


key = os.environ.get("GROQ_API_KEY", "")
if key:
    print(f"✓ Key loaded: {key[:4]}...{key[-4:]}")
else:
    print("✗ Key not found")

✓ API key loaded from Colab secrets
✓ Key loaded: gsk_...UcG9


In [ ]:
# ==============================================================================
# COUNTERFACTUAL ROBUSTNESS EVALUATION
# ==============================================================================

import os
import json
from typing import Dict, List, Any, Tuple
from tqdm import tqdm

# ------------------------------------------------------------------------------
# Counterfactual Robustness: Core Logic
# ------------------------------------------------------------------------------

def evaluate_counterfactual_robustness(
    groq_client,
    dataset: List[Dict[str, Any]],
    model_name: str,
    result_file: str,
    checkpoint_interval: int = 10
) -> Dict[str, Any]:
    """
    Evaluate counterfactual robustness capability.

    This tests whether LLMs can:
    1. Detect factual errors in provided documents
    2. Correct the errors using their own knowledge

    Key differences:
    - Uses en_fact.json dataset (100 instances)
    - Tests model's ability to detect false information
    - Requires model to have prior knowledge of the answer
    - Documents contain 'positive_wrong' (counterfactual) documents

    Args:
        groq_client: Initialized Groq LLM client
        dataset: The en_fact.json dataset
        model_name: Name of the model being evaluated
        result_file: Path to save results
        checkpoint_interval: How often to save checkpoints

    Returns:
        Dictionary with ACC, ACCdoc, ED, and CR metrics
    """

    logger.info(f"Starting Counterfactual Robustness evaluation for {model_name}")
    logger.info(f"Total instances: {len(dataset)}")

    # Load existing results if any
    processed_results = load_existing_results(result_file)

    # Metrics trackers
    total_instances = 0
    acc_without_docs = 0  # Can answer correctly without documents
    acc_with_docs = 0      # Can answer correctly with counterfactual documents
    error_detected = 0     # Detected factual errors in documents
    error_corrected = 0    # Corrected errors after detection

    for idx, instance in enumerate(tqdm(dataset, desc="Counterfactual")):
        instance_id = instance["id"]

        # Skip if already processed
        if str(instance_id) in processed_results:
            result = processed_results[str(instance_id)]
            total_instances += 1
            if result.get("correct_without_docs", False):
                acc_without_docs += 1
            if result.get("correct_with_docs", False):
                acc_with_docs += 1
            if result.get("error_detected", False):
                error_detected += 1
                if result.get("error_corrected", False):
                    error_corrected += 1
            continue

        query = instance["query"]
        ground_truth = instance["answer"]

        # STEP 1: Test if model knows answer WITHOUT documents (baseline)
        try:
            prompt_no_docs = {
                "system": RGB_PROMPT_EN["system"],
                "user": f"Question:\n{query}"
            }
            response_no_docs = groq_client.generate(prompt_no_docs)
            correct_without_docs = check_answer_correctness(response_no_docs, ground_truth)
        except Exception as e:
            logger.error(f"Error in no-docs test for instance {instance_id}: {str(e)}")
            correct_without_docs = False
            response_no_docs = ""

        # STEP 2: Test WITH counterfactual documents
        # Use positive_wrong documents (contain factual errors)
        counterfactual_docs = instance.get("positive_wrong", [])

        # Sample up to 5 counterfactual documents
        selected_docs = counterfactual_docs[:5] if len(counterfactual_docs) > 0 else []

        # Format prompt with counterfactual documents
        prompt_with_docs = format_rgb_prompt(query, selected_docs)

        try:
            response_with_docs = groq_client.generate(prompt_with_docs)

            # Check if model detected the error
            detected_error = check_error_detection(response_with_docs)

            # Check if model provided correct answer despite wrong documents
            correct_with_docs = check_answer_correctness(response_with_docs, ground_truth)

            # If error was detected, check if it was corrected
            corrected_error = False
            if detected_error:
                corrected_error = correct_with_docs

        except Exception as e:
            logger.error(f"Error processing instance {instance_id}: {str(e)}")
            response_with_docs = ""
            detected_error = False
            correct_with_docs = False
            corrected_error = False

        # Save result
        result_record = {
            "id": instance_id,
            "query": query,
            "ground_truth": ground_truth,
            "response_without_docs": response_no_docs,
            "response_with_docs": response_with_docs,
            "correct_without_docs": correct_without_docs,
            "correct_with_docs": correct_with_docs,
            "error_detected": detected_error,
            "error_corrected": corrected_error,
            "model": model_name,
            "num_docs": len(selected_docs)
        }

        append_result(result_file, result_record)

        # Update counters
        total_instances += 1
        if correct_without_docs:
            acc_without_docs += 1
        if correct_with_docs:
            acc_with_docs += 1
        if detected_error:
            error_detected += 1
            if corrected_error:
                error_corrected += 1

        # Checkpoint logging
        if (idx + 1) % checkpoint_interval == 0:
            logger.info(
                f"Checkpoint {idx + 1}/{len(dataset)} | "
                f"ACC: {(acc_without_docs/total_instances)*100:.2f}% | "
                f"ACCdoc: {(acc_with_docs/total_instances)*100:.2f}%"
            )

    # Calculate final metrics
    acc = (acc_without_docs / total_instances) * 100 if total_instances > 0 else 0.0
    acc_doc = (acc_with_docs / total_instances) * 100 if total_instances > 0 else 0.0
    ed_rate = (error_detected / total_instances) * 100 if total_instances > 0 else 0.0
    cr_rate = (error_corrected / error_detected) * 100 if error_detected > 0 else 0.0

    metrics = {
        "ACC": round(acc, 2),                    # Accuracy without documents
        "ACCdoc": round(acc_doc, 2),            # Accuracy with counterfactual docs
        "ED": round(ed_rate, 2),                # Error detection rate
        "CR": round(cr_rate, 2),                # Error correction rate
        "total_instances": total_instances,
        "acc_without_docs_count": acc_without_docs,
        "acc_with_docs_count": acc_with_docs,
        "error_detected_count": error_detected,
        "error_corrected_count": error_corrected,
        "model": model_name
    }

    logger.info(f"Counterfactual Robustness Results for {model_name}:")
    logger.info(f"  ACC (without docs): {acc:.2f}%")
    logger.info(f"  ACCdoc (with counterfactual docs): {acc_doc:.2f}%")
    logger.info(f"  Error Detection Rate: {ed_rate:.2f}%")
    logger.info(f"  Error Correction Rate: {cr_rate:.2f}%")

    return metrics


# ------------------------------------------------------------------------------
# Error Detection Logic
# ------------------------------------------------------------------------------

def check_error_detection(response: str) -> bool:
    """
    Check if the model detected factual errors in the documents.

    According to RGB paper, the model should output:
    "There are factual errors in the provided documents."

    Args:
        response: The model's generated response

    Returns:
        True if error detection phrase is present
    """
    response_lower = response.lower()

    # Define error detection indicators
    detection_indicators = [
        "there are factual errors",
        "factual errors in the provided documents",
        "factual errors in the documents",
        "documents contain factual errors",
        "documents have factual errors",
        "incorrect information in the documents",
        "false information in the documents",
        "documents are incorrect",
        "documents are wrong",
        "error in the document",
        "mistake in the document"
    ]

    # Check if any detection indicator is present
    for indicator in detection_indicators:
        if indicator in response_lower:
            return True

    return False


# ------------------------------------------------------------------------------
# Answer Correctness Check
# ------------------------------------------------------------------------------

def check_answer_correctness(response: str, ground_truth: List[List[str]]) -> bool:
    """
    Check if response contains the correct answer.

    Args:
        response: Model's generated response
        ground_truth: List of acceptable answer variants

    Returns:
        True if any answer variant is present in response
    """
    response_lower = response.lower()

    # Flatten the answer list (if nested)
    if isinstance(ground_truth, list):
        if len(ground_truth) > 0 and isinstance(ground_truth[0], list):
            # Nested list - flatten it
            all_answers = [ans for sublist in ground_truth for ans in sublist]
        else:
            all_answers = ground_truth
    else:
        all_answers = [ground_truth]

    # Check if any answer variant appears in response
    for answer_variant in all_answers:
        if str(answer_variant).lower() in response_lower:
            return True

    return False


# ------------------------------------------------------------------------------
# Run Counterfactual Robustness Experiment
# ------------------------------------------------------------------------------

def run_counterfactual_robustness_experiment(
    api_key: str,
    model_configs: List[Dict[str, str]],
    dataset_path: str,
    results_dir: str
):
    """
    Run counterfactual robustness evaluation for multiple models.

    Args:
        api_key: Groq API key
        model_configs: List of dicts with 'name' and 'model_id' keys
        dataset_path: Path to en_fact.json
        results_dir: Directory to save results
    """

    # Load dataset
    dataset = load_jsonl(dataset_path)
    logger.info(f"Loaded {len(dataset)} instances from {dataset_path}")

    all_results = {}

    for model_config in model_configs:
        model_short_name = model_config["name"]
        model_id = model_config["model_id"]

        logger.info(f"\n{'='*80}")
        logger.info(f"Evaluating model: {model_short_name} ({model_id})")
        logger.info(f"{'='*80}\n")

        # Initialize Groq client for this model
        groq_client = GroqLLMClient(
            api_key=api_key,
            model_name=model_id,
            temperature=0.0,
            max_retries=3,
            retry_backoff_sec=5
        )

        # Create result file path
        result_file = os.path.join(
            results_dir,
            f"counterfactual_robustness_{model_short_name}.jsonl"
        )

        # Run evaluation
        metrics = evaluate_counterfactual_robustness(
            groq_client=groq_client,
            dataset=dataset,
            model_name=model_short_name,
            result_file=result_file
        )

        all_results[model_short_name] = metrics

    # Save summary
    summary_file = os.path.join(results_dir, "counterfactual_robustness_summary.json")
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    # Print summary table
    logger.info(f"\n{'='*80}")
    logger.info("COUNTERFACTUAL ROBUSTNESS - SUMMARY TABLE")
    logger.info(f"{'='*80}\n")

    print("\n| Model | ACC | ACCdoc | ED | CR |")
    print("|-------|-----|--------|----|----|")
    for model_name, metrics in all_results.items():
        print(f"| {model_name} | {metrics['ACC']:.0f} | {metrics['ACCdoc']:.0f} | "
              f"{metrics['ED']:.0f} | {metrics['CR']:.2f} |")

    return all_results


# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

# Define your models
MODEL_CONFIGS = [
    {"name": "qwen3_32b", "model_id": "qwen/qwen3-32b"},
    {"name": "llama3_1_8b", "model_id": "llama-3.1-8b-instant"},
    {"name": "gemma2_9b", "model_id": "gemma2-9b-it"}
]

# Your Groq API key
GROQ_API_KEY = key  # Replace with actual key

# Paths
DATASET_PATH_FACT = os.path.join(DATASET_DIR, "en_fact.json")
RESULTS_DIR_COUNTER = os.path.join(RESULTS_DIR, "counterfactual_robustness")
os.makedirs(RESULTS_DIR_COUNTER, exist_ok=True)

# Run experiment
results = run_counterfactual_robustness_experiment(
    api_key=GROQ_API_KEY,
    model_configs=MODEL_CONFIGS,
    dataset_path=DATASET_PATH_FACT,
    results_dir=RESULTS_DIR_COUNTER
)


2026-01-18 04:32:44 | INFO | Loaded 100 instances from /content/drive/MyDrive/data/en_fact.json


INFO:cap_25_rgbeval:Loaded 100 instances from /content/drive/MyDrive/data/en_fact.json


2026-01-18 04:32:44 | INFO | 


INFO:cap_25_rgbeval:


2026-01-18 04:32:44 | INFO | Evaluating model: qwen3_32b (qwen/qwen3-32b)


...

2026-01-18 04:54:47 | ERROR | Error in no-docs test for instance 5: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}


ERROR:cap_25_rgbeval:Error in no-docs test for instance 5: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}


2026-01-18 04:54:47 | WARNING | Groq inference failed (attempt 1/3): Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}


Counterfactual:   5%|▌         | 5/100 [01:54<36:20, 22.95s/it]


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# COUNTERFACTUAL ROBUSTNESS RESULTS ANALYSIS AND REPORTING
# ==============================================================================

import os
import json
import pandas as pd
from typing import Dict, List, Any

# ------------------------------------------------------------------------------
# Load Counterfactual Robustness Results
# ------------------------------------------------------------------------------

def load_counterfactual_robustness_results(results_dir: str) -> Dict[str, List[Dict]]:
    """
    Load all counterfactual robustness JSONL result files.

    Returns:
        Dictionary mapping model names to their results
    """
    all_results = {}

    if not os.path.exists(results_dir):
        print(f"Error: Directory {results_dir} does not exist")
        return all_results

    for filename in os.listdir(results_dir):
        if filename.startswith("counterfactual_robustness_") and filename.endswith(".jsonl"):
            filepath = os.path.join(results_dir, filename)

            # Extract model name: counterfactual_robustness_{model_name}.jsonl
            model_name = filename.replace("counterfactual_robustness_", "").replace(".jsonl", "")

            # Load JSONL file
            results = []
            with open(filepath, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        results.append(json.loads(line))

            all_results[model_name] = results
            print(f"✓ Loaded {len(results)} results for {model_name}")

    return all_results


# ------------------------------------------------------------------------------
# Calculate Counterfactual Metrics
# ------------------------------------------------------------------------------

def calculate_counterfactual_metrics(results: List[Dict]) -> Dict[str, Any]:
    """
    Calculate all counterfactual robustness metrics.

    Metrics:
    - ACC: Accuracy without documents (baseline knowledge)
    - ACCdoc: Accuracy with counterfactual documents
    - ED: Error Detection rate
    - CR: Error Correction rate (among detected errors)

    Args:
        results: List of result dictionaries

    Returns:
        Dictionary with calculated metrics
    """
    total_instances = len(results)

    # Count different outcomes
    correct_without_docs = sum(1 for r in results if r.get("correct_without_docs", False))
    correct_with_docs = sum(1 for r in results if r.get("correct_with_docs", False))
    error_detected = sum(1 for r in results if r.get("error_detected", False))
    error_corrected = sum(1 for r in results if r.get("error_corrected", False))

    # Calculate percentages
    acc = (correct_without_docs / total_instances * 100) if total_instances > 0 else 0.0
    acc_doc = (correct_with_docs / total_instances * 100) if total_instances > 0 else 0.0
    ed = (error_detected / total_instances * 100) if total_instances > 0 else 0.0
    cr = (error_corrected / error_detected * 100) if error_detected > 0 else 0.0

    return {
        "ACC": round(acc, 2),
        "ACCdoc": round(acc_doc, 2),
        "ED": round(ed, 2),
        "CR": round(cr, 2),
        "total_instances": total_instances,
        "correct_without_docs_count": correct_without_docs,
        "correct_with_docs_count": correct_with_docs,
        "error_detected_count": error_detected,
        "error_corrected_count": error_corrected
    }


# ------------------------------------------------------------------------------
# Generate Summary Table
# ------------------------------------------------------------------------------

def generate_counterfactual_summary_table(all_results: Dict[str, List[Dict]]) -> pd.DataFrame:
    """
    Generate summary table with all counterfactual metrics.

    Args:
        all_results: Dictionary mapping model names to their results

    Returns:
        DataFrame with summary statistics
    """
    summary_data = []

    for model_name, results in all_results.items():
        metrics = calculate_counterfactual_metrics(results)
        summary_data.append({
            "Model": model_name,
            "ACC": metrics["ACC"],
            "ACCdoc": metrics["ACCdoc"],
            "ED": metrics["ED"],
            "CR": metrics["CR"]
        })

    # Create DataFrame and sort by ACC (descending)
    df = pd.DataFrame(summary_data)
    df = df.sort_values("ACC", ascending=False)

    return df


# ------------------------------------------------------------------------------
# Detailed Breakdown
# ------------------------------------------------------------------------------

def print_detailed_breakdown(all_results: Dict[str, List[Dict]]):
    """Print detailed breakdown for each model."""

    print("="*80)
    print("DETAILED BREAKDOWN BY MODEL")
    print("="*80)
    print()

    for model_name, results in all_results.items():
        metrics = calculate_counterfactual_metrics(results)

        print(f"Model: {model_name}")
        print("-" * 60)
        print(f"  Total Instances: {metrics['total_instances']}")
        print()
        print(f"  ACC (Accuracy without documents):")
        print(f"    - Correct: {metrics['correct_without_docs_count']}/{metrics['total_instances']}")
        print(f"    - Percentage: {metrics['ACC']:.2f}%")
        print()
        print(f"  ACCdoc (Accuracy with counterfactual documents):")
        print(f"    - Correct: {metrics['correct_with_docs_count']}/{metrics['total_instances']}")
        print(f"    - Percentage: {metrics['ACCdoc']:.2f}%")
        print()
        print(f"  ED (Error Detection):")
        print(f"    - Detected: {metrics['error_detected_count']}/{metrics['total_instances']}")
        print(f"    - Detection Rate: {metrics['ED']:.2f}%")
        print()
        print(f"  CR (Error Correction among detected):")
        print(f"    - Corrected: {metrics['error_corrected_count']}/{metrics['error_detected_count']}")
        print(f"    - Correction Rate: {metrics['CR']:.2f}%")
        print()


# ------------------------------------------------------------------------------
# Analyze Error Patterns
# ------------------------------------------------------------------------------

def analyze_error_patterns(all_results: Dict[str, List[Dict]]):
    """Analyze common error patterns across models."""

    print("="*80)
    print("ERROR PATTERN ANALYSIS")
    print("="*80)
    print()

    for model_name, results in all_results.items():
        print(f"Model: {model_name}")
        print("-" * 60)

        # Categorize instances
        knew_answer_kept_correct = []
        knew_answer_misled = []
        didnt_know_detected_error = []
        didnt_know_accepted_wrong = []

        for r in results:
            knew_without = r.get("correct_without_docs", False)
            correct_with = r.get("correct_with_docs", False)
            detected = r.get("error_detected", False)

            if knew_without and correct_with:
                knew_answer_kept_correct.append(r)
            elif knew_without and not correct_with:
                knew_answer_misled.append(r)
            elif not knew_without and detected:
                didnt_know_detected_error.append(r)
            elif not knew_without and not detected:
                didnt_know_accepted_wrong.append(r)

        print(f"  Knew answer & kept correct: {len(knew_answer_kept_correct)}")
        print(f"  Knew answer but was misled: {len(knew_answer_misled)}")
        print(f"  Didn't know but detected error: {len(didnt_know_detected_error)}")
        print(f"  Didn't know & accepted wrong info: {len(didnt_know_accepted_wrong)}")
        print()


# ------------------------------------------------------------------------------
# Compare with RGB Paper Benchmarks
# ------------------------------------------------------------------------------

def compare_with_rgb_paper(summary_df: pd.DataFrame):
    """Compare results with RGB paper benchmarks."""

    print("="*80)
    print("COMPARISON WITH RGB PAPER BENCHMARKS")
    print("="*80)
    print()

    # RGB paper results (Table 7 - English)
    rgb_benchmarks = {
        "ChatGPT-en": {"ACC": 89, "ACCdoc": 9, "ED": 8, "CR": 57.14}
    }

    print("RGB Paper Results (English):")
    print("| Model | ACC | ACCdoc | ED | CR |")
    print("|-------|-----|--------|----|----|")
    for model, scores in rgb_benchmarks.items():
        print(f"| {model} | {scores['ACC']:.0f} | {scores['ACCdoc']:.0f} | "
              f"{scores['ED']:.0f} | {scores['CR']:.2f} |")

    print()
    print("Your Results:")
    print(summary_df.to_markdown(index=False))
    print()

    print("Note: RGB paper had limited counterfactual results.")
    print("Lower ACCdoc is expected - it shows the model resists wrong information.")
    print("Higher ED and CR are better - model detects and corrects errors.")
    print()


# ------------------------------------------------------------------------------
# Export Functions
# ------------------------------------------------------------------------------

def export_summary_to_csv(df: pd.DataFrame, output_file: str):
    """Export summary table to CSV."""
    df.to_csv(output_file, index=False)
    print(f"✓ Summary exported to {output_file}")


def export_summary_to_markdown(df: pd.DataFrame, output_file: str):
    """Export summary table to Markdown format."""
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("# Counterfactual Robustness Results\n\n")
        f.write("## Metrics Explanation\n\n")
        f.write("- **ACC**: Accuracy without documents (baseline knowledge)\n")
        f.write("- **ACCdoc**: Accuracy with counterfactual documents (lower is better - resists misinformation)\n")
        f.write("- **ED**: Error Detection rate (higher is better)\n")
        f.write("- **CR**: Error Correction rate among detected errors (higher is better)\n\n")
        f.write("## Results\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n")
    print(f"✓ Markdown summary exported to {output_file}")


def export_detailed_analysis(all_results: Dict[str, List[Dict]], output_dir: str):
    """Export detailed analysis for each model."""

    for model_name, results in all_results.items():
        metrics = calculate_counterfactual_metrics(results)

        # Categorize instances for detailed analysis
        analysis = {
            "model": model_name,
            "metrics": metrics,
            "error_detected_instances": [],
            "error_corrected_instances": [],
            "misled_instances": [],
            "correct_despite_wrong_docs": []
        }

        for r in results:
            if r.get("error_detected", False):
                analysis["error_detected_instances"].append({
                    "id": r["id"],
                    "query": r["query"],
                    "response": r["response_with_docs"][:200]  # First 200 chars
                })

            if r.get("error_corrected", False):
                analysis["error_corrected_instances"].append({
                    "id": r["id"],
                    "query": r["query"]
                })

            if r.get("correct_without_docs") and not r.get("correct_with_docs"):
                analysis["misled_instances"].append({
                    "id": r["id"],
                    "query": r["query"]
                })

            if r.get("correct_with_docs"):
                analysis["correct_despite_wrong_docs"].append({
                    "id": r["id"],
                    "query": r["query"]
                })

        # Save to file
        output_file = os.path.join(output_dir, f"{model_name}_detailed_analysis.json")
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(analysis, f, indent=2, ensure_ascii=False)

        print(f"✓ Detailed analysis exported for {model_name}")


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

# Set your results directory path
RESULTS_DIR_COUNTER = os.path.join(RESULTS_DIR, "counterfactual_robustness")

# Load all results
print("Loading counterfactual robustness results...\n")
all_model_results = load_counterfactual_robustness_results(RESULTS_DIR_COUNTER)

if len(all_model_results) == 0:
    print("No results found! Check your results directory path.")
else:
    print(f"\nFound results for {len(all_model_results)} models\n")

    # Generate summary table
    summary_df = generate_counterfactual_summary_table(all_model_results)

    # Display the table
    print("="*80)
    print("COUNTERFACTUAL ROBUSTNESS SUMMARY TABLE")
    print("="*80)
    print()
    print(summary_df.to_string(index=False))
    print()

    # Display in markdown format (for your report)
    print("="*80)
    print("MARKDOWN FORMAT (Copy this for your report)")
    print("="*80)
    print()
    print(summary_df.to_markdown(index=False))
    print()

    # Save to files
    summary_csv = os.path.join(RESULTS_DIR_COUNTER, "summary_table.csv")
    summary_md = os.path.join(RESULTS_DIR_COUNTER, "summary_table.md")

    export_summary_to_csv(summary_df, summary_csv)
    export_summary_to_markdown(summary_df, summary_md)

    # Detailed breakdown
    print_detailed_breakdown(all_model_results)

    # Error pattern analysis
    analyze_error_patterns(all_model_results)

    # Compare with RGB paper
    compare_with_rgb_paper(summary_df)

    # Export detailed analysis
    print("="*80)
    print("EXPORTING DETAILED ANALYSIS")
    print("="*80)
    print()
    export_detailed_analysis(all_model_results, RESULTS_DIR_COUNTER)

    print()
    print("="*80)
    print("REPORT GENERATION COMPLETE")
    print("="*80)


Loading counterfactual robustness results...

✓ Loaded 101 results for qwen3_32b
✓ Loaded 100 results for llama3_1_8b
✓ Loaded 5 results for gemma2_9b

Found results for 3 models

COUNTERFACTUAL ROBUSTNESS SUMMARY TABLE

      Model   ACC  ACCdoc    ED    CR
llama3_1_8b 84.00    36.0 36.00 88.89
  qwen3_32b 78.22    49.5 74.26 65.33
  gemma2_9b  0.00     0.0  0.00  0.00

MARKDOWN FORMAT (Copy this for your report)

| Model       |   ACC |   ACCdoc |    ED |    CR |
|:------------|------:|---------:|------:|------:|
| llama3_1_8b | 84    |     36   | 36    | 88.89 |
| qwen3_32b   | 78.22 |     49.5 | 74.26 | 65.33 |
| gemma2_9b   |  0    |      0   |  0    |  0    |

✓ Summary exported to /content/drive/MyDrive/results_01/counterfactual_robustness/summary_table.csv
✓ Markdown summary exported to /content/drive/MyDrive/results_01/counterfactual_robustness/summary_table.md
DETAILED BREAKDOWN BY MODEL

Model: qwen3_32b
------------------------------------------------------------
  Total In

In [ ]:
# Load Groq API key from Colab secrets
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("✓ Groq API key loaded successfully")
    else:
        print("✗ API key is empty. Please add it to Colab secrets.")
except Exception as e:
    print(f"✗ Error loading API key: {e}")
    print("Please add GROQ_API_KEY to Colab secrets (click 🔑 in sidebar)")

✓ Groq API key loaded successfully


In [ ]:
# ==============================================================================
# NOISE ROBUSTNESS EVALUATION
# ==============================================================================

import os
import json
import random
from typing import Dict, List, Any, Tuple
from tqdm import tqdm

# ------------------------------------------------------------------------------
# Noise Robustness: Document Selection Logic
# ------------------------------------------------------------------------------

def select_documents_for_noise_robustness(
    instance: Dict[str, Any],
    noise_ratio: float,
    passage_num: int = 5
) -> Tuple[List[str], str]:
    """
    Select documents for noise robustness task.

    Key concept:
    - Uses positive documents (contain the answer) + negative documents (noise)
    - Noise ratio determines proportion of irrelevant documents
    - Tests if model can find answer among distracting information

    Args:
        instance: Data instance from en_refine.json
        noise_ratio: Ratio of noise documents (0.0, 0.2, 0.4, 0.6, 0.8)
        passage_num: Total number of documents to provide (default: 5)

    Returns:
        Tuple of (selected_documents, document_composition_info)
    """
    positive_docs = instance["positive"]  # Documents containing the answer
    negative_docs = instance["negative"]  # Irrelevant/noise documents

    # Calculate how many of each type
    num_noise = int(passage_num * noise_ratio)
    num_positive = passage_num - num_noise

    selected_docs = []

    # Select positive documents (contain answer)
    if num_positive > 0 and len(positive_docs) > 0:
        sample_count = min(num_positive, len(positive_docs))
        selected_docs.extend(random.sample(positive_docs, sample_count))

    # Select negative documents (noise)
    if num_noise > 0 and len(negative_docs) > 0:
        sample_count = min(num_noise, len(negative_docs))
        selected_docs.extend(random.sample(negative_docs, sample_count))

    # Shuffle to randomize order (model shouldn't rely on document position)
    random.shuffle(selected_docs)

    # Create composition info for logging
    composition = f"Positive={len([d for d in selected_docs if d in positive_docs])}, Noise={len([d for d in selected_docs if d in negative_docs])}"

    return selected_docs, composition


# ------------------------------------------------------------------------------
# Answer Checking for Noise Robustness
# ------------------------------------------------------------------------------

def check_noise_robustness_answer(response: str, ground_truth: List[List[str]]) -> bool:
    """
    Check if response contains the correct answer.

    For noise robustness, answer is typically a list of lists with answer variants.

    Args:
        response: Model's generated response
        ground_truth: List of acceptable answer variants
                     e.g., [["January 2, 2022", "Jan 2, 2022", "2022-01-02"]]

    Returns:
        True if any answer variant is found in response
    """
    response_lower = response.lower()

    # Flatten nested lists if needed
    all_answers = []
    for item in ground_truth:
        if isinstance(item, list):
            all_answers.extend(item)
        else:
            all_answers.append(item)

    # Check if any answer variant appears in response
    for answer_variant in all_answers:
        if str(answer_variant).lower() in response_lower:
            return True

    return False


# ------------------------------------------------------------------------------
# Noise Robustness Evaluation
# ------------------------------------------------------------------------------

def evaluate_noise_robustness(
    groq_client,
    dataset: List[Dict[str, Any]],
    model_name: str,
    noise_ratio: float,
    result_file: str,
    checkpoint_interval: int = 10
) -> Dict[str, Any]:
    """
    Evaluate noise robustness capability at a specific noise ratio.

    Args:
        groq_client: Initialized Groq LLM client
        dataset: The en_refine.json dataset (300 instances)
        model_name: Name of the model being evaluated
        noise_ratio: Noise ratio (0.0, 0.2, 0.4, 0.6, 0.8)
        result_file: Path to save results
        checkpoint_interval: How often to save checkpoints

    Returns:
        Dictionary with accuracy metrics
    """

    logger.info(f"Starting Noise Robustness evaluation for {model_name} at noise_ratio={noise_ratio}")
    logger.info(f"Total instances: {len(dataset)}")

    # Load existing results if any
    processed_results = load_existing_results(result_file)

    total_instances = 0
    correct_count = 0

    for idx, instance in enumerate(tqdm(dataset, desc=f"NoiseRobust (noise={noise_ratio})")):
        instance_id = instance["id"]

        # Skip if already processed
        if str(instance_id) in processed_results:
            result = processed_results[str(instance_id)]
            total_instances += 1
            if result.get("correct", False):
                correct_count += 1
            continue

        query = instance["query"]
        ground_truth = instance["answer"]

        # Select documents for this noise ratio
        selected_docs, composition = select_documents_for_noise_robustness(
            instance, noise_ratio, passage_num=5
        )

        # Format prompt
        prompt = format_rgb_prompt(query, selected_docs)

        # Generate response
        try:
            response = groq_client.generate(prompt)

            # Check if answer is correct
            is_correct = check_noise_robustness_answer(response, ground_truth)

            # Save result
            result_record = {
                "id": instance_id,
                "query": query,
                "ground_truth": ground_truth,
                "response": response,
                "correct": is_correct,
                "noise_ratio": noise_ratio,
                "doc_composition": composition,
                "model": model_name,
                "num_docs": len(selected_docs)
            }

            append_result(result_file, result_record)

            total_instances += 1
            if is_correct:
                correct_count += 1

            # Checkpoint logging
            if (idx + 1) % checkpoint_interval == 0:
                current_acc = (correct_count / total_instances) * 100
                logger.info(
                    f"Checkpoint {idx + 1}/{len(dataset)} | "
                    f"Accuracy: {current_acc:.2f}%"
                )

        except Exception as e:
            logger.error(f"Error processing instance {instance_id}: {str(e)}")
            continue

    # Calculate final metrics
    accuracy = (correct_count / total_instances) * 100 if total_instances > 0 else 0.0

    metrics = {
        "accuracy": accuracy,
        "total_correct": correct_count,
        "total_instances": total_instances,
        "noise_ratio": noise_ratio,
        "model": model_name
    }

    logger.info(f"Noise Robustness Results for {model_name} (noise={noise_ratio}):")
    logger.info(f"  Accuracy: {accuracy:.2f}%")
    logger.info(f"  Correct: {correct_count}/{total_instances}")

    return metrics


# ------------------------------------------------------------------------------
# Run Noise Robustness Experiment (All Noise Ratios)
# ------------------------------------------------------------------------------

def run_noise_robustness_experiment(
    api_key: str,
    model_configs: List[Dict[str, str]],
    dataset_path: str,
    results_dir: str,
    noise_ratios: List[float] = [0.0, 0.2, 0.4, 0.6, 0.8]
):
    """
    Run noise robustness evaluation for multiple models and noise ratios.

    Args:
        api_key: Groq API key
        model_configs: List of dicts with 'name' and 'model_id' keys
        dataset_path: Path to en_refine.json
        results_dir: Directory to save results
        noise_ratios: List of noise ratios to test (default: [0.0, 0.2, 0.4, 0.6, 0.8])
    """

    # Load dataset
    dataset = load_jsonl(dataset_path)
    logger.info(f"Loaded {len(dataset)} instances from {dataset_path}")

    all_results = {}

    for model_config in model_configs:
        model_short_name = model_config["name"]
        model_id = model_config["model_id"]

        logger.info(f"\n{'='*80}")
        logger.info(f"Evaluating model: {model_short_name} ({model_id})")
        logger.info(f"{'='*80}\n")

        # Initialize Groq client for this model
        groq_client = GroqLLMClient(
            api_key=api_key,
            model_name=model_id,
            temperature=0.0,
            max_retries=3,
            retry_backoff_sec=5
        )

        model_results = {}

        # Evaluate at each noise ratio
        for noise_ratio in noise_ratios:
            logger.info(f"\n--- Noise Ratio: {noise_ratio} ---")

            # Create result file path
            result_file = os.path.join(
                results_dir,
                f"noise_robustness_{model_short_name}_noise_{noise_ratio}.jsonl"
            )

            # Run evaluation for this noise ratio
            metrics = evaluate_noise_robustness(
                groq_client=groq_client,
                dataset=dataset,
                model_name=model_short_name,
                noise_ratio=noise_ratio,
                result_file=result_file
            )

            model_results[f"noise_{noise_ratio}"] = metrics

        all_results[model_short_name] = model_results

    # Save summary
    summary_file = os.path.join(results_dir, "noise_robustness_summary.json")
    with open(summary_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    # Print summary table
    logger.info(f"\n{'='*80}")
    logger.info("NOISE ROBUSTNESS - SUMMARY TABLE")
    logger.info(f"{'='*80}\n")

    print("\n| Model | Noise 0.0 | Noise 0.2 | Noise 0.4 | Noise 0.6 | Noise 0.8 |")
    print("|-------|-----------|-----------|-----------|-----------|-----------|")
    for model_name, results in all_results.items():
        acc_0 = results.get("noise_0.0", {}).get("accuracy", 0)
        acc_2 = results.get("noise_0.2", {}).get("accuracy", 0)
        acc_4 = results.get("noise_0.4", {}).get("accuracy", 0)
        acc_6 = results.get("noise_0.6", {}).get("accuracy", 0)
        acc_8 = results.get("noise_0.8", {}).get("accuracy", 0)
        print(f"| {model_name} | {acc_0:.2f} | {acc_2:.2f} | {acc_4:.2f} | {acc_6:.2f} | {acc_8:.2f} |")

    return all_results


# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

# Define your models
# Define your models
MODEL_CONFIGS = [
    {"name": "llama3_1_8b", "model_id": "llama-3.1-8b-instant"},
    {"name": "llama3_3_70b", "model_id": "llama-3.3-70b-versatile"},
    {"name": "qwen3_32b", "model_id": "qwen/qwen3-32b"}
]



# Paths
DATASET_PATH_REFINE = os.path.join(DATASET_DIR, "en_refine.json")
RESULTS_DIR_NOISE = os.path.join(RESULTS_DIR, "noise_robustness")
os.makedirs(RESULTS_DIR_NOISE, exist_ok=True)

# Run experiment with all noise ratios
results = run_noise_robustness_experiment(
    api_key=GROQ_API_KEY,
    model_configs=MODEL_CONFIGS,
    dataset_path=DATASET_PATH_REFINE,
    results_dir=RESULTS_DIR_NOISE,
    noise_ratios=[0.0, 0.4, 0.8]
)


2026-01-24 22:00:39 | INFO | Loaded 300 instances from /content/drive/MyDrive/data/en_refine.json


INFO:cap_25_rgbeval:Loaded 300 instances from /content/drive/MyDrive/data/en_refine.json


2026-01-24 22:00:39 | INFO | 


INFO:cap_25_rgbeval:


2026-01-24 22:00:39 | INFO | Evaluating model: llama3_1_8b (llama-3.1-8b-instant)


...

2026-01-24 22:20:20 | INFO | NOISE ROBUSTNESS - SUMMARY TABLE


INFO:cap_25_rgbeval:NOISE ROBUSTNESS - SUMMARY TABLE


2026-01-24 22:20:20 | INFO | ================================================================================



INFO:cap_25_rgbeval:================================================================================




| Model | Noise 0.0 | Noise 0.2 | Noise 0.4 | Noise 0.6 | Noise 0.8 |
|-------|-----------|-----------|-----------|-----------|-----------|
| llama3_1_8b | 99.00 | 0.00 | 98.33 | 0.00 | 93.33 |
| llama3_3_70b | 99.67 | 0.00 | 100.00 | 0.00 | 97.00 |
| qwen3_32b | 100.00 | 0.00 | 100.00 | 0.00 | 99.33 |


In [ ]:
# ==============================================================================
# NOISE ROBUSTNESS RESULTS ANALYSIS AND REPORTING
# ==============================================================================

import os
import json
import pandas as pd
import numpy as np
from typing import Dict, List, Any
import matplotlib.pyplot as plt

# ------------------------------------------------------------------------------
# Load Noise Robustness Results
# ------------------------------------------------------------------------------

def load_noise_robustness_results(results_dir: str) -> Dict[str, Dict[str, List[Dict]]]:
    """
    Load all noise robustness JSONL result files.

    Returns:
        Nested dict: {model_name: {noise_ratio: [results]}}
    """
    all_results = {}

    if not os.path.exists(results_dir):
        print(f"Error: Directory {results_dir} does not exist")
        return all_results

    for filename in os.listdir(results_dir):
        if filename.startswith("noise_robustness_") and filename.endswith(".jsonl"):
            filepath = os.path.join(results_dir, filename)

            # Parse filename: noise_robustness_{model_name}_noise_{noise_ratio}.jsonl
            parts = filename.replace("noise_robustness_", "").replace(".jsonl", "").split("_noise_")

            if len(parts) == 2:
                model_name = parts[0]
                noise_ratio = parts[1]

                # Load JSONL file
                results = []
                with open(filepath, "r", encoding="utf-8") as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            results.append(json.loads(line))

                # Initialize model dict if needed
                if model_name not in all_results:
                    all_results[model_name] = {}

                all_results[model_name][f"noise_{noise_ratio}"] = results
                print(f"✓ Loaded {len(results)} results for {model_name} at noise={noise_ratio}")

    return all_results


# ------------------------------------------------------------------------------
# Calculate Accuracy Metrics
# ------------------------------------------------------------------------------

def calculate_accuracy_metrics(results: List[Dict]) -> Dict[str, Any]:
    """
    Calculate accuracy metrics from noise robustness results.
    """
    total_instances = len(results)
    correct_count = sum(1 for r in results if r.get("correct", False))

    accuracy = (correct_count / total_instances * 100) if total_instances > 0 else 0.0

    return {
        "total_instances": total_instances,
        "total_correct": correct_count,
        "total_incorrect": total_instances - correct_count,
        "accuracy": round(accuracy, 2)
    }


# ------------------------------------------------------------------------------
# Generate Summary Table
# ------------------------------------------------------------------------------

def generate_noise_robustness_summary_table(
    all_results: Dict[str, Dict[str, List[Dict]]]
) -> pd.DataFrame:
    """
    Generate summary table with accuracy across all noise ratios.
    """
    summary_data = []

    for model_name, noise_results in all_results.items():
        row_data = {"Model": model_name}

        # Get accuracy for each noise ratio
        for noise_key in ["noise_0.0", "noise_0.2", "noise_0.4", "noise_0.6", "noise_0.8"]:
            if noise_key in noise_results:
                metrics = calculate_accuracy_metrics(noise_results[noise_key])
                row_data[f"Noise {noise_key.split('_')[1]}"] = metrics["accuracy"]
            else:
                row_data[f"Noise {noise_key.split('_')[1]}"] = None

        summary_data.append(row_data)

    # Create DataFrame and sort by Noise 0.0 accuracy (descending)
    df = pd.DataFrame(summary_data)
    if "Noise 0.0" in df.columns:
        df = df.sort_values("Noise 0.0", ascending=False)

    return df


# ------------------------------------------------------------------------------
# Calculate Performance Degradation
# ------------------------------------------------------------------------------

def calculate_performance_degradation(all_results: Dict[str, Dict[str, List[Dict]]]):
    """
    Calculate how much accuracy degrades as noise increases.
    """
    print("="*80)
    print("PERFORMANCE DEGRADATION ANALYSIS")
    print("="*80)
    print()

    for model_name, noise_results in all_results.items():
        print(f"Model: {model_name}")
        print("-" * 60)

        # Get baseline (noise 0.0)
        baseline_metrics = calculate_accuracy_metrics(noise_results.get("noise_0.0", []))
        baseline_acc = baseline_metrics["accuracy"]

        print(f"  Baseline (Noise 0.0): {baseline_acc:.2f}%")
        print()

        # Calculate degradation for each noise level
        for noise_key in ["noise_0.2", "noise_0.4", "noise_0.6", "noise_0.8"]:
            if noise_key in noise_results:
                metrics = calculate_accuracy_metrics(noise_results[noise_key])
                current_acc = metrics["accuracy"]

                # Calculate absolute and relative degradation
                abs_degradation = baseline_acc - current_acc
                rel_degradation = (abs_degradation / baseline_acc * 100) if baseline_acc > 0 else 0

                noise_value = noise_key.split("_")[1]
                print(f"  Noise {noise_value}:")
                print(f"    - Accuracy: {current_acc:.2f}%")
                print(f"    - Absolute Drop: {abs_degradation:.2f} percentage points")
                print(f"    - Relative Drop: {rel_degradation:.2f}% from baseline")

        print()


# ------------------------------------------------------------------------------
# Detailed Breakdown
# ------------------------------------------------------------------------------

def print_detailed_breakdown(all_results: Dict[str, Dict[str, List[Dict]]]):
    """Print detailed breakdown for each model and noise ratio."""

    print("="*80)
    print("DETAILED BREAKDOWN BY MODEL AND NOISE RATIO")
    print("="*80)
    print()

    for model_name, noise_results in all_results.items():
        print(f"Model: {model_name}")
        print("-" * 60)

        for noise_key in sorted(noise_results.keys()):
            results = noise_results[noise_key]
            metrics = calculate_accuracy_metrics(results)
            noise_value = noise_key.split("_")[1]

            print(f"  Noise Ratio {noise_value}:")
            print(f"    - Total Instances: {metrics['total_instances']}")
            print(f"    - Correct: {metrics['total_correct']}")
            print(f"    - Incorrect: {metrics['total_incorrect']}")
            print(f"    - Accuracy: {metrics['accuracy']:.2f}%")

        print()


# ------------------------------------------------------------------------------
# Compare with RGB Paper Benchmarks
# ------------------------------------------------------------------------------

def compare_with_rgb_paper(summary_df: pd.DataFrame):
    """Compare results with RGB paper benchmarks."""

    print("="*80)
    print("COMPARISON WITH RGB PAPER BENCHMARKS")
    print("="*80)
    print()

    # RGB paper results (Table 1 - English)
    rgb_benchmarks = {
        "ChatGPT": {"0.0": 96.33, "0.2": 94.67, "0.4": 94.00, "0.6": 90.00, "0.8": 76.00},
        "Qwen-7B-Chat": {"0.0": 94.33, "0.2": 91.67, "0.4": 91.00, "0.6": 87.67, "0.8": 73.67},
        "Vicuna-7B-v1.3": {"0.0": 87.67, "0.2": 83.33, "0.4": 86.00, "0.6": 82.33, "0.8": 60.33},
        "ChatGLM-6B": {"0.0": 93.67, "0.2": 90.67, "0.4": 89.33, "0.6": 84.67, "0.8": 70.67},
        "ChatGLM2-6B": {"0.0": 91.33, "0.2": 89.67, "0.4": 83.00, "0.6": 77.33, "0.8": 57.33},
        "BELLE-7B-2M": {"0.0": 83.33, "0.2": 81.00, "0.4": 79.00, "0.6": 71.33, "0.8": 64.67}
    }

    print("RGB Paper Best Results (English):")
    print("| Model | Noise 0.0 | Noise 0.2 | Noise 0.4 | Noise 0.6 | Noise 0.8 |")
    print("|-------|-----------|-----------|-----------|-----------|-----------|")
    for model, scores in rgb_benchmarks.items():
        print(f"| {model} | {scores['0.0']:.2f} | {scores['0.2']:.2f} | "
              f"{scores['0.4']:.2f} | {scores['0.6']:.2f} | {scores['0.8']:.2f} |")

    print()
    print("Your Results:")
    print(summary_df.to_markdown(index=False))
    print()


# ------------------------------------------------------------------------------
# Robustness Score
# ------------------------------------------------------------------------------

def calculate_robustness_score(all_results: Dict[str, Dict[str, List[Dict]]]):
    """
    Calculate a robustness score based on how well accuracy is maintained.
    Lower degradation = higher robustness.
    """
    print("="*80)
    print("ROBUSTNESS SCORE")
    print("="*80)
    print()

    print("Robustness Score = Average accuracy across all noise levels")
    print("Higher score = Better noise robustness")
    print()

    scores = []

    for model_name, noise_results in all_results.items():
        accuracies = []

        for noise_key in ["noise_0.0", "noise_0.2", "noise_0.4", "noise_0.6", "noise_0.8"]:
            if noise_key in noise_results:
                metrics = calculate_accuracy_metrics(noise_results[noise_key])
                accuracies.append(metrics["accuracy"])

        avg_accuracy = np.mean(accuracies) if accuracies else 0

        scores.append({
            "Model": model_name,
            "Average Accuracy": round(avg_accuracy, 2),
            "Robustness Score": round(avg_accuracy, 2)
        })

        print(f"Model: {model_name}")
        print(f"  - Average Accuracy: {avg_accuracy:.2f}%")
        print(f"  - Individual: {', '.join([f'{a:.2f}%' for a in accuracies])}")
        print()

    # Sort by robustness score
    scores_df = pd.DataFrame(scores).sort_values("Robustness Score", ascending=False)
    print("Ranking:")
    print(scores_df.to_markdown(index=False))
    print()


# ------------------------------------------------------------------------------
# Export Functions
# ------------------------------------------------------------------------------

def export_summary_to_csv(df: pd.DataFrame, output_file: str):
    """Export summary table to CSV."""
    df.to_csv(output_file, index=False)
    print(f"✓ Summary exported to {output_file}")


def export_summary_to_markdown(df: pd.DataFrame, output_file: str):
    """Export summary table to Markdown format."""
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("# Noise Robustness Results\n\n")
        f.write("## Results Table\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n\n")
        f.write("## Interpretation\n\n")
        f.write("- **Noise 0.0**: No noise (all documents are relevant)\n")
        f.write("- **Noise 0.2**: 20% of documents are irrelevant\n")
        f.write("- **Noise 0.4**: 40% of documents are irrelevant\n")
        f.write("- **Noise 0.6**: 60% of documents are irrelevant\n")
        f.write("- **Noise 0.8**: 80% of documents are irrelevant\n\n")
        f.write("Higher accuracy at higher noise ratios indicates better robustness.\n")
    print(f"✓ Markdown summary exported to {output_file}")


def export_detailed_results(all_results: Dict[str, Dict[str, List[Dict]]], output_dir: str):
    """Export detailed analysis for each model."""

    for model_name, noise_results in all_results.items():
        model_summary = {
            "model": model_name,
            "noise_ratios": {}
        }

        for noise_key, results in noise_results.items():
            metrics = calculate_accuracy_metrics(results)

            # Get correct and incorrect instances
            correct_instances = [r for r in results if r.get("correct", False)]
            incorrect_instances = [r for r in results if not r.get("correct", False)]

            model_summary["noise_ratios"][noise_key] = {
                "metrics": metrics,
                "sample_correct": [
                    {"id": r["id"], "query": r["query"]}
                    for r in correct_instances[:5]
                ],
                "sample_incorrect": [
                    {"id": r["id"], "query": r["query"]}
                    for r in incorrect_instances[:5]
                ]
            }

        # Save to file
        output_file = os.path.join(output_dir, f"{model_name}_detailed_analysis.json")
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(model_summary, f, indent=2, ensure_ascii=False)

        print(f"✓ Detailed analysis exported for {model_name}")


# ==============================================================================
# MAIN EXECUTION
# ==============================================================================

# Set your results directory path
RESULTS_DIR_NOISE = os.path.join(RESULTS_DIR, "noise_robustness")

# Load all results
print("Loading noise robustness results...\n")
all_model_results = load_noise_robustness_results(RESULTS_DIR_NOISE)

if len(all_model_results) == 0:
    print("No results found! Check your results directory path.")
else:
    print(f"\nFound results for {len(all_model_results)} models\n")

    # Generate summary table
    summary_df = generate_noise_robustness_summary_table(all_model_results)

    # Display the table
    print("="*80)
    print("NOISE ROBUSTNESS SUMMARY TABLE")
    print("="*80)
    print()
    print(summary_df.to_string(index=False))
    print()

    # Display in markdown format (for your report)
    print("="*80)
    print("MARKDOWN FORMAT (Copy this for your report)")
    print("="*80)
    print()
    print(summary_df.to_markdown(index=False))
    print()

    # Save to files
    summary_csv = os.path.join(RESULTS_DIR_NOISE, "summary_table.csv")
    summary_md = os.path.join(RESULTS_DIR_NOISE, "summary_table.md")

    export_summary_to_csv(summary_df, summary_csv)
    export_summary_to_markdown(summary_df, summary_md)

    # Detailed breakdown
    print_detailed_breakdown(all_model_results)

    # Performance degradation analysis
    calculate_performance_degradation(all_model_results)

    # Robustness score
    calculate_robustness_score(all_model_results)

    # Compare with RGB paper
    compare_with_rgb_paper(summary_df)

    # Export detailed analysis
    print("="*80)
    print("EXPORTING DETAILED ANALYSIS")
    print("="*80)
    print()
    export_detailed_results(all_model_results, RESULTS_DIR_NOISE)

    print()
    print("="*80)
    print("REPORT GENERATION COMPLETE")
    print("="*80)


Loading noise robustness results...

✓ Loaded 300 results for llama3_1_8b at noise=0.0
✓ Loaded 300 results for llama3_1_8b at noise=0.4
✓ Loaded 300 results for llama3_1_8b at noise=0.8
✓ Loaded 300 results for llama3_3_70b at noise=0.0
✓ Loaded 300 results for llama3_3_70b at noise=0.4
✓ Loaded 300 results for llama3_3_70b at noise=0.8
✓ Loaded 300 results for qwen3_32b at noise=0.0
✓ Loaded 300 results for qwen3_32b at noise=0.4
✓ Loaded 300 results for qwen3_32b at noise=0.8

Found results for 3 models

NOISE ROBUSTNESS SUMMARY TABLE

       Model  Noise 0.0 Noise 0.2  Noise 0.4 Noise 0.6  Noise 0.8
   qwen3_32b     100.00      None     100.00      None      99.33
llama3_3_70b      99.67      None     100.00      None      97.00
 llama3_1_8b      99.00      None      98.33      None      93.33

MARKDOWN FORMAT (Copy this for your report)

| Model        |   Noise 0.0 | Noise 0.2   |   Noise 0.4 | Noise 0.6   |   Noise 0.8 |
|:-------------|------------:|:------------|------------:|